In [2]:

import json
from collections import Counter
from datetime import datetime, timedelta
from itertools import product, zip_longest
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
import plotly.graph_objects as goa
import regex
import requests
import yaml
from plotly.colors import qualitative, sample_colorscale
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
from datetime import datetime
import pytz
from src.utils import (
    guardarExcel,
    guardarExcelMulti
)

from datetime import timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.api import getHistoricoMOW
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime,
)
from src.utils import (
    isEmpty,
    loadEstaciones,
    loadLocalizaciones,
    localizeFecha,
    parallelizeFunction,
    rellenarId,
    removeDoubleQuotes,
    splitDataframe,
)
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isEmpty,
    map_cod2name,
    map_name2use_name,
    parallelizeFunction,
    setEF
)
from src.utils.util import (
    loadEstaciones
)
from src.processor import (
    XPECProcessor
    
)
from src.utils.topos import getEstacionamientos

In [3]:
from datetime import timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.api import getHistoricoMOW
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime,
)
# pd.set_option("future.no_silent_downcasting", True)
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

import argparse
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import regex
import yaml
from tqdm.auto import tqdm

from src.api import cargarHistorico, getHistoricoMOW
from src.processor import LogProcessor
from src.utils import isValidCode, parallelizeFunction, parseDate, rellenarId
from src.api.APIs import (
    getEstadoCirculacionesTecnicas,
    getPlanificacionCirculacionesTecnicas)

In [4]:
# Tipos de tren que queremos
train_types = {
    "Approach": "APROXIMACIÓN",
    "Arrival": "LLEGADA",
    "Departure": "SALIDA",
    "Elimination": "SUPRESIÓN",
    "End": "FIN",
    "Entry": "ENTRY",
    "Exit": "EXIT",
    "Maneuver": "MANIOBRA",
    "Platform": "ORIGEN",
    "PlatformForecast": "PREVISIÓN",  # "PREDICCIÓN",
    "Stopped": "STOP",
    "TrackingLost": "LOST_TRACK",
}

# Orden lógico de movimientos
mov_sorter = {
    v: k
    for k, v in enumerate(
        [
            "PREVISIÓN",
            "APROXIMACIÓN",
            "MANIOBRALLEGADA",
            "EXIT",
            "LLEGADA",
            "FIN",
            "BAJA",
            "ORIGEN",
            "SALIDA",
            "MANIOBRASALIDA",
            "MANIOBRA",
        ]
    )
}
from src.processor import  SitraProcessor

In [5]:
def cargarHistorico(
    start_date: str,
    end_date: str,
    estaciones: list[str],
    trenes: list[str],
    xSIV: bool,
    JCTC: bool,
    pro: bool = True
):
    # Comprobamos que la fecha de fin sea después de la de inicio
    if end_date <= start_date:
        end_date = (pd.to_datetime(start_date) + timedelta(days=1)).strftime(
            "%Y-%m-%d %H:%M:%S"
        )

    historico = getHistoricoMOW(
        estaciones=estaciones, trenes=trenes, inicio=start_date, fin=end_date, pro=pro,xSIV=xSIV, jCTC=JCTC
    )
    historico = historico[
        (historico["Fecha"] >= pd.to_datetime(start_date))
        & (historico["Fecha"] <= pd.to_datetime(end_date))
    ]
    # Usamos movimientos auditados
    #historico = historico[
         #np.invert(historico["FuenteVía"].isin(["PLANNED", "SITRA_PROVIDED"]))
     #]
    historico = historico[historico["NTécnico"].apply(isValidCode)].dropna(
        subset=["Movimiento"]
    )
    # historico["mov_ord"] = historico["Movimiento"].apply(mov_sorter.get)
    return historico

In [6]:

ntrenes = [rellenarId(f"{i}") for i in np.arange(100000)]
start_date = "2026-02-20"
end_date = "2026-02-26"
estaciones =["05000"]
historico_pro = cargarHistorico(start_date,end_date,estaciones,ntrenes,xSIV=False,JCTC=True,pro=True)

1771542000000
incio: 1771542000.0
fin: 1772060400.0
Página 0 (2026-02-20 - 2026-02-26)

Formateando fechas.:   0%|          | 0/2 [00:00<?, ?it/s]

In [7]:
historico_pro.sort_values(by="Fecha")

,Fecha,NTécnico,CTC,Mnemónico,Nombre,Código,Elemento,Sentido,Movimiento,FuenteMovimiento,FuenteMensaje,Descripción
193,2026-02-20 00:00:29,02226,ANT,GRA,Granada,05000,A18O,IZQUIERDA,SALIDA,CTC_MSE,JCTC,"{'train': '02226', 'occupation': False, 'eleme..."
194,2026-02-20 00:00:29,02226,ANT,GRA,Granada,05000,5O,IZQUIERDA,LLEGADA,CTC_MSE,JCTC,"{'train': '02226', 'occupation': True, 'elemen..."
2710,2026-02-20 00:28:52,97228,ANT,GRA,Granada,05000,AE4O,IZQUIERDA,SALIDA,CTC_MSE,JCTC,"{'train': '97228', 'occupation': False, 'eleme..."
2711,2026-02-20 00:28:52,97228,ANT,GRA,Granada,05000,E4O,IZQUIERDA,LLEGADA,CTC_MSE,JCTC,"{'train': '97228', 'occupation': True, 'elemen..."
2714,2026-02-20 00:29:34,97228,ANT,GRA,Granada,05000,A18O,IZQUIERDA,LLEGADA,CTC_MSE,JCTC,"{'train': '97228', 'occupation': True, 'elemen..."
...,...,...,...,...,...,...,...,...,...,...,...,...
1677,2026-02-25 22:19:28,13127,ANT,GRA,Granada,05000,A2O,DERECHA,LLEGADA,CTC_MSE,JCTC,"{'train': '13127', 'occupation': True, 'elemen..."
1678,2026-02-25 22:19:38,13127,ANT,GRA,Granada,05000,A2O,DERECHA,SALIDA,CTC_MSE,JCTC,"{'train': '13127', 'occupation': False, 'eleme..."
1679,2026-02-25 22:19:38,13127,ANT,GRA,Granada,05000,E2O,DERECHA,LLEGADA,CTC_MSE,JCTC,"{'train': '13127', 'occupation': True, 'elemen..."
1681,2026-02-25 22:19:52,13127,ANT,GRA,Granada,05000,E2O,DERECHA,SALIDA,CTC_MSE,JCTC,"{'train': '13127', 'occupation': False, 'eleme..."


In [8]:
start_date = "2026-02-20"
end_date = "2026-02-26"
xsiv = cargarHistorico(start_date,end_date,[],ntrenes,xSIV=True,JCTC=False,pro=True)

1771542000000
incio: 1771542000.0
fin: 1772060400.0
Página 274 (2026-02-20 - 2026-02-26)

Formateando fechas.:   0%|          | 0/2570 [00:00<?, ?it/s]

In [9]:
df = historico_pro.copy()

In [55]:
df.sort_values("Fecha")

,Elemento,Fecha,NTécnico,CTC,Mnemónico,Nombre,Código,Sentido,Movimiento,FuenteMovimiento,FuenteMensaje,Descripción
42,5,2026-02-20 00:00:29,02226,ANT,GRA,Granada,05000,IZQUIERDA,LLEGADA,CTC_MSE,JCTC,"{'train': '02226', 'occupation': True, 'elemen..."
43,5,2026-02-20 06:03:39,02226,ANT,GRA,Granada,05000,IZQUIERDA,SALIDA,CTC_MSE,JCTC,"{'train': '02226', 'occupation': True, 'elemen..."
90,6,2026-02-20 06:03:49,08394,ANT,GRA,Granada,05000,IZQUIERDA,SALIDA,CTC_MSE,JCTC,"{'train': '08394', 'occupation': True, 'elemen..."
303,4,2026-02-20 06:10:56,97242,ANT,GRA,Granada,05000,IZQUIERDA,LLEGADA,CTC_MSE,JCTC,"{'train': '97242', 'occupation': True, 'elemen..."
304,4,2026-02-20 06:11:58,97242,ANT,GRA,Granada,05000,IZQUIERDA,SALIDA,CTC_MSE,JCTC,"{'train': '97242', 'occupation': True, 'elemen..."
...,...,...,...,...,...,...,...,...,...,...,...,...
256,1,2026-02-25 21:14:18,13134,ANT,GRA,Granada,05000,IZQUIERDA,LLEGADA,CTC_MSE,JCTC,"{'train': '13134', 'occupation': True, 'elemen..."
257,1,2026-02-25 21:16:06,13134,ANT,GRA,Granada,05000,IZQUIERDA,SALIDA,CTC_MSE,JCTC,"{'train': '13134', 'occupation': True, 'elemen..."
212,1,2026-02-25 21:16:06,13127,ANT,GRA,Granada,05000,IZQUIERDA,LLEGADA,CTC_MSE,JCTC,"{'train': '13127', 'occupation': True, 'elemen..."
99,6,2026-02-25 22:08:40,08394,ANT,GRA,Granada,05000,IZQUIERDA,LLEGADA,CTC_MSE,JCTC,"{'train': '08394', 'occupation': True, 'elemen..."


In [56]:
xsiv.sort_values("Fecha")

,Fecha,NTécnico,CTC,Nombre,Código,Secuencia,Movimiento,Elemento,FuenteMovimiento,Vía,...,FechaOrigen,LíneaComercial,CódigoOrigen,NombreOrigen,CódigoDestino,NombreDestino,SalidaPlanificada,FuenteMensaje,Descripción,Núcleo
2464839,2026-02-20 00:00:00,91729,MAC,VALLECAS-INDUSTRIAL,70001,1.0,PÉRDIDA_SEGUIMIENTO,None,None,None,...,2026-02-19,None,70001,VALLECAS-INDUSTRIAL,60004,MADRID-SANTA CATALINA,2026-02-19 23:00:00,XSIV,{},None
2486012,2026-02-20 00:00:00,93712,COR,ALCOLEA DE CORDOBA,50413,1.0,PÉRDIDA_SEGUIMIENTO,None,None,None,...,2026-02-19,None,50413,ALCOLEA DE CORDOBA,50500,CORDOBA-JULIO ANGUITA,2026-02-19 23:00:00,XSIV,{},None
953625,2026-02-20 00:00:01,22132,OVI,VILLABONA-TABLADIELLO (APD),15305,6.0,SALIDA,XX_EP2,CTC_MIE,2,...,2026-02-19,C1,15410,GIJON-SANZ CRESPO,15218,LLAMAQUIQUE (APD),2026-02-19 23:45:00,XSIV,{},ASTURIAS
953624,2026-02-20 00:00:01,22132,OVI,VILLABONA DE ASTURIAS,15301,7.0,PREVISIÓN,PROYECCIÓN,CTC_MIE,2,...,2026-02-19,C1,15410,GIJON-SANZ CRESPO,15218,LLAMAQUIQUE (APD),2026-02-19 23:48:20,XSIV,{},ASTURIAS
953627,2026-02-20 00:00:01,22132,OVI,LUGO DE LLANERA,15300,8.0,PREVISIÓN,PROYECCIÓN,CTC_MIE,2,...,2026-02-19,C1,15410,GIJON-SANZ CRESPO,15218,LLAMAQUIQUE (APD),2026-02-19 23:53:20,XSIV,{},ASTURIAS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102865,2026-02-26 00:00:00,02226,ANT,BIF. LA CHANA,B0110,10.0,SALIDA,E2O,CTC_MIE,1,...,2026-02-25,None,02003,ANTEQUERA-SANTA ANA,05000,GRANADA,2026-02-25 23:11:20,XSIV,{},None
102850,2026-02-26 00:00:00,02226,ANT,BIF. LA CHANA,B0110,10.0,SALIDA,E2O,CTC_MIE,1,...,2026-02-25,None,02003,ANTEQUERA-SANTA ANA,05000,GRANADA,2026-02-25 23:11:20,XSIV,{},None
102849,2026-02-26 00:00:00,02226,ANT,BIF. LA CHANA,B0110,10.0,SALIDA,E2O,CTC_MIE,1,...,2026-02-25,None,02003,ANTEQUERA-SANTA ANA,05000,GRANADA,2026-02-25 23:11:20,XSIV,{},None
102852,2026-02-26 00:00:00,02226,ANT,BIF. LA CHANA,B0110,10.0,SALIDA,E2O,CTC_MIE,1,...,2026-02-25,None,02003,ANTEQUERA-SANTA ANA,05000,GRANADA,2026-02-25 23:11:20,XSIV,{},None


In [57]:
estacion= "05000"

In [13]:
topo = getEstacionamientos(["05000"])
map_estacion_elemento = (
        topo[["Código", "Vía"]]
        .dropna()
        .groupby("Código")
        .agg(lambda x: {el: [] for el in set(x)})
        .to_dict()["Vía"]
    )
topo = topo[["VíaTécnica","Vía"]]
topo.rename(columns={"VíaTécnica":"Elemento"},inplace=True)

In [14]:

df = pd.merge(
topo,
df,
on=["Elemento"],
how="right"
)
df = df.dropna(subset=["Vía"]).reset_index(drop=True)
df.drop(columns="Elemento",inplace=True)
df.rename(columns={"Vía":"Elemento"},inplace=True)


In [58]:
df_logs = df[(df["Código"] == estacion)].copy()
df_logs["mov_ord"] = df_logs["Movimiento"].apply(mov_sorter.get)
df_logs = df_logs.sort_values(by=["NTécnico", "Fecha", "mov_ord"])

In [59]:
if estacion not in map_cod2name:
    aux = df_logs[["Código", "Nombre"]].dropna().drop_duplicates()
    if aux.empty:
       print("No hay datos")
    map_cod2name.update(dict(aux.values))
    name = map_cod2name[estacion]
    use_name = regex.sub(r"[^\w\s-\(\)]", "", name.lower())
    use_name = regex.sub(r"[\s-]+", "_", use_name)
    map_name2use_name.update({name: use_name})

In [60]:
from typing import Union
import pandas as pd
from typing import Union


def filterTrains(
    df: pd.DataFrame,
    day: Union[str, pd.Timestamp] = None,
    platform: str = None,
    station: str = None,
):
    """
    Devuelve un dataframe con los datos de un día en una vía en una estación
    """

    filt_df = df.copy()

    if day is not None:
        day = pd.to_datetime(day)
        filt_df = filt_df[filt_df["Fecha"].dt.date == day.date()]

    if station is not None:
        filt_df = filt_df[filt_df["Código"] == station]

    if platform is not None:
        trenes = filt_df[filt_df["Elemento"] == platform]["NTécnico"].unique()
        filt_df = filt_df[
            (filt_df["NTécnico"].isin(trenes))
            & ((filt_df["Elemento"] == platform) | (filt_df["Vía"].isna()))
        ]

    return filt_df
def splitTrainsByDate(
        df: pd.DataFrame, 
        day: Union[str, pd.Timestamp] = None,
        platform: str = None,
        station: str = None,
        hour_diff: int = 1,
        filter_mov: bool = False,
    ):
        """
        Separa los datos de un tren en una vía un día concreto
        hour_diff hace que los trenes del mismo número separados por más de estas horas se cuenten por separado
        """
        filt_df = filterTrains(
            df=df,
            day=day,
            platform=platform,
            station=station,
        )
        if filt_df.empty:
            return None, None

        # Agrupar trenes por número técnico y fecha.
        filt_df = filt_df.sort_values(by=["NTécnico", "Fecha","mov_ord"]).reset_index(
            drop=True
        )
        filt_df["tdiff"] = filt_df["Fecha"].apply(pd.to_datetime, dayfirst=True).diff()
        # No incluimos la columna tdiff
        t_cols = filt_df.columns[:-1]
        df_split = np.split(
            filt_df[t_cols],
            np.where(
                (~(filt_df["tdiff"] < timedelta(hours=hour_diff)))
                | (~filt_df["NTécnico"].eq(filt_df["NTécnico"].shift()))
                | (~filt_df["Elemento"].eq(filt_df["Elemento"].shift()))
            )[0][1:],
        )

        # Como las finalizaciones no tienen vía, nos aseguramos de que no se asigna una vía
        # a un fin si no hay llegada.
        col_fecha = np.where(t_cols == "Fecha")[0][0]
        if not filter_mov:
            return sorted(df_split, key=lambda x: x.iloc[-1]["NTécnico"]), t_cols
        filt_split = []
        for sp in df_split:
            aux_df = pd.DataFrame(sp, columns=t_cols)
            movs = "_".join(aux_df["Movimiento"])
            if (("FIN" in movs) or ("BAJA" in movs)) and not ("LLEGADA" in movs):
                # wrong_fin = np.in1d(sp[:, 1], ["FIN", "BAJA"])
                wrong_fin = np.in1d(aux_df["Movimiento"].values, ["FIN", "BAJA"])
                sp = np.delete(sp, wrong_fin, axis=0)
                if not sp.size:
                    continue
            filt_split.append(sp)
        filt_split = sorted(filt_split, key=lambda x: x[-1][col_fecha])

        return filt_split, t_cols

In [61]:
df_split, t_cols = splitTrainsByDate(
        df=df_logs,
        # platform=via,
        station=estacion,
        hour_diff=12,
        filter_mov=False,
    )

In [62]:
lista_con_id = [df for df in df_split if (df["NTécnico"] == "10086").any()]


In [63]:
lista_con_id[1]

,Elemento,Fecha,NTécnico,CTC,Mnemónico,Nombre,Código,Sentido,Movimiento,FuenteMovimiento,FuenteMensaje,Descripción,mov_ord
124,4,2026-02-20 10:51:18,10086,ANT,GRA,Granada,05000,IZQUIERDA,LLEGADA,CTC_MSE,JCTC,"{'train': '10086', 'occupation': True, 'elemen...",4
125,4,2026-02-20 10:52:58,10086,ANT,GRA,Granada,05000,IZQUIERDA,SALIDA,CTC_MSE,JCTC,"{'train': '10086', 'occupation': True, 'elemen...",8


In [21]:
producto = xsiv[["NTécnico",'CategoríaCirculación', 'Producto', 'Empresa']].copy()

In [22]:
if not df_split:
    display("no existe trenes")
for df_t in df_split:
    if df_t.empty:
        display("esta vacio")
    df_t = pd.merge(
        producto,
        df_t,
        on=["NTécnico"],
        how = "right"
    )
    df_t.drop_duplicates(subset=["NTécnico", "Fecha", "Elemento"],inplace=True)
    cols = list(df_t.columns)
    if "Fecha" in cols:
        cols.insert(0, cols.pop(cols.index("Fecha")))
        df_t = df_t[cols]
    df_t["tdiff"] = df_t["Fecha"].diff().dt.total_seconds()
    df_t["mdiff"] = df_t["mov_ord"].diff()
    df_t = df_t.reset_index(drop=True).reset_index()
    df_t["prev_index"] = df_t["index"].shift(fill_value=-1)
    for i, row in df_t[1:].iterrows():
        if (row["tdiff"] < 5) & (row["mdiff"] < 0):
            row = row.copy()
            aux_row = df_t.loc[row["prev_index"]].copy()
            mdate = [row["Fecha"], aux_row["Fecha"]]
            aux_row["Fecha"] = max(mdate)
            row["Fecha"] = min(mdate)
            df_t.loc[row["prev_index"]] = row
            df_t.loc[i] = aux_row
    df_t_split = np.split(
        df_t,
        np.where((~df_t["Elemento"].eq(df_t["Elemento"].shift())))[0][1:],
            )
    apariciones = len(df_t_split)
    for n, train_day in enumerate(df_t_split, 1):
        # Si no hay ninguna vía pasamos
        if (train_day["Elemento"].isna().all()) or (
            train_day["Movimiento"]
            .apply(
                lambda x: x
                in [
                    "APROXIMACIÓN",
                    "PREVISIÓN",
                    "ALTA",
                ]
            )
            .all()
        ):
            continue
        # Si aparece en otra vía, creamos una "salida" provisional a la hora que aparece en la otra
        if n < apariciones and not train_day.iloc[-1]["Movimiento"] == "SALIDA":
            # Comprobamos que la siguiente aparición no sea aproximación
            next_aparicion = df_t_split[n].dropna(subset=["Elemento"])
            next_aparicion = next_aparicion[
                np.invert(
                    next_aparicion["Movimiento"].apply(
                        lambda x: x
                        in [
                            "APROXIMACIÓN",
                            "PREVISIÓN",
                            # "ALTA",
                        ]
                    )
                )
            ]
            if not next_aparicion.empty:
                new_row = train_day.iloc[-1:].copy()
                next_aparicion = next_aparicion.iloc[0]
                # new_row["Fecha"] = next_aparicion["Fecha"]
                new_row["Movimiento"] = "CAMBIO_VÍA"
                new_row["cambio_vía"] = [
                    {
                        "Fecha": next_aparicion["Fecha"],
                        "Elemento": next_aparicion["Elemento"],
                    }
                ]
                # # Asignamos la fecha de la siguiente aparición
                # new_fecha = next_aparicion.iloc[0]["Fecha"]
                # new_row["Fecha"] = new_fecha
                train_day = pd.concat([train_day, new_row])
        via = train_day["Elemento"].dropna().iloc[0]
        map_estacion_elemento[estacion][via].append(train_day)


In [23]:
def getMovementType(t1: str, t2: str, mov_ts: str):
    llegada = r"(MANIOBRA→|PREVISIÓN→|APROXIMACIÓN→)*(MANIOBRA→|LLEGADA→)"
    fin = r"(FIN(→)?|ELIMINACIÓN(→)?|SUPRESIÓN(→)?|BAJA(→)?)(MANIOBRA(→)?|CAMBIO_VÍA)*"
    alta = r"(ALTA→)"
    salida = r"(MANIOBRA(→)?|SALIDA(→)?|CAMBIO_VÍA(→)?)"
    # opt_maniobra = r"(MANIOBRA.*?(→)?)"
    if t1 == t2:
        if regex.search(
            rf"^{llegada}+{salida}+$",
            mov_ts,
        ):
            return "PASO"
        elif regex.search(
            rf"^{alta}+{salida}+$",
            mov_ts,
        ):
            return "ORIGEN"
        elif regex.search(
            rf"^{llegada}+{fin}+$",
            mov_ts,
        ):
            return "FIN"
        else:
            return "INCOMPLETO"
    else:
        if regex.search(
            rf"^{llegada}+({fin}+{alta}+)+{salida}+$",
            mov_ts,
        ):
            return "ROTACIÓN"
        elif regex.search(
            rf"^{llegada}+{fin}+{salida}+$",
            mov_ts,
        ):
            return "FIN"
        elif regex.search(
            rf"^({alta}{fin})*{alta}+{salida}+$",
            mov_ts,
        ):
            return "ORIGEN"
        elif regex.search(
            rf"^{llegada}+{alta}+{salida}+$",
            mov_ts,
        ):
            return "RENOMBRADO"
        else:
            return "ROTACIÓN_INCORRECTA"

In [24]:
PRODUCTO_VACIO = [
    "material vacío",
    "material vacio",
    "material vacío ram",
    "material vacio ram",
    "servicio interno",
]
def procesarSecuenciasEstacion(
        map_vias: dict[str, list[pd.DataFrame]],
        mov_sorter: dict[str, int],
    ):
        """
        Procesa las secuencias de movimientos en una estacion para cada vía
        map_vias: Dataframe con las siguientes columnas:
            - Fecha
            - NTécnico
            - Vía
            - TipoVía
            - Movimiento
        """

        datos = []
        cols = (
            # Trenes
            ["T1", "T2", "T_seq","Código","ProductoT1", "ProductoT2", "EF", "Vía", "TipoVía"]
            # Planificación
            + ["LlegadaPlanificada", "SalidaPlanificada", "OcupaciónPlanificada"]
            # Secuencias de movimientos
            + ["Movimiento", "Mov_seq", "full_seq", "cambio_vía"]
            # Anticipación
            + [
                "Anticipación",
                "AnticipaciónPlataforma",
                "AnticipaciónSalida",
                "AnticipaciónAproximación",
                "AnticipaciónLlegada",
            ]
            # Ocupación
            + ["InicioOcupación", "FinOcupación", "Ocupación"]
            # # Tiempos totales
            # + ["Inicio", "Fin", "TiempoTotal"]
            # Todos los movimientos posibles
            + list(mov_sorter.keys())
        )

        for Elemento, trains in map_vias.items():
            if not trains:
                continue
            # *TODO*: Procesar primero cada tren por separado y luego unirlos
            # De esta manera evitamos que haya, por ejemplo, aproximaciones entre la llegada y salida de uno anterior:
            # [llegada tren 1 -> aproximación tren 2 -> salida tren 1] -> [llegada tren 2 -> salida tren 2]
            # pasaría a ser: [llegada tren 1 -> salida tren 1] -> [aproximación tren 2 -> llegada tren 2 -> salida tren 2]

            # Componemos los trenes que han pasado por la vía
            df_via = pd.concat(trains).sort_values(by=["Fecha", "mov_ord"])
            # df_via = df_via[
            #     np.invert(df_via["Movimiento"].isin(["APROXIMACIÓN", "PREVISIÓN"]))
            # ]

            # df_via["_prods"] = df_via["Producto"].apply(lambda x: [x, "Material Vacio"])
            # Separamos si hay salida o si el siguiente tren es de otro tipo
            cortes = np.where(
                (df_via["Movimiento"].shift().isin(["SALIDA", "CAMBIO_VÍA"]))
                | (~df_via["NTécnico"].shift().eq(df_via["NTécnico"]))
            )[0]

            via_split = np.split(df_via, cortes)
            for train_day in via_split:
                if train_day.empty:
                    continue
                # Si unicamente es una aproximación lo tratamos de forma diferente
                if (
                    train_day["Movimiento"]
                    .apply(lambda x: x in ["APROXIMACIÓN", "PREVISIÓN", "ALTA"])
                    .all()
                ):
                    continue
                info = {c: pd.NA for c in cols}
                # # Limpiamos los movimientos de sobra
                # train_day = train_day.drop_duplicates(
                #     subset=[
                #         "Movimiento",
                #         "Producto",
                #         "FechaOrigen",
                #         "NTécnico",
                #         "Código",
                #         "Vía",
                #     ]
                # )

                # Números técnicos
                info["T1"], info["ProductoT1"] = train_day.iloc[0][
                    ["NTécnico", "Producto"]
                ].values
                info["T2"], info["ProductoT2"] = train_day.iloc[-1][
                    ["NTécnico", "Producto"]
                ].values
                if "Código" in train_day.columns:
                    codigo = train_day["Código"].dropna()
                    info["Código"] = codigo.iloc[0] if not codigo.empty else pd.NA
                vals = train_day["NTécnico"].values
                seq = [vals[0]]
                for v in vals[1:]:
                    if not v == seq[-1]:
                        seq.append(v)
                info["T_seq"] = "→".join(seq)
                if "Empresa" not in train_day.columns:
                    info["EF"] = setEF(
                        "".join(
                            [
                                el if el and pd.notna(el) else ""
                                for el in [info["ProductoT1"], info["ProductoT2"]]
                            ]
                        )
                    )
                else:
                    empresa = train_day["Empresa"].dropna()
                    if not empresa.empty:
                        info["EF"] = empresa.iloc[0]
                    else:
                        info["EF"] = pd.NA
                    # info["EF"] = train_day["Empresa"].dropna().iloc[0]
                    info["Elemento"] = Elemento

                # Movimientos registrados
                for mtype, mdate in (
                    train_day[["Movimiento", "Fecha"]]
                    .drop_duplicates(subset=["Movimiento"])
                    .values
                ):
                    info[mtype] = mdate

                # Secuencia de movimientos registrados
                vals = train_day[["NTécnico", "Movimiento"]].values
                full_seq = [vals[0].tolist()]
                mov_seq = [vals[0][1]]
                for n, v in vals[1:]:
                    # Si el movimiento del tren es igual que el anterior, lo ignoro
                    if n == full_seq[-1][0] and v == full_seq[-1][1]:
                        continue
                    full_seq.append([n, v])
                    mov_seq.append(v)
                info["full_seq"] = full_seq
                info["Mov_seq"] = "→".join(mov_seq)

                # Comprobamos los cambios de vía y usamos el último (debería ser único)
                if "cambio_vía" in train_day.columns:
                    cambio = train_day["cambio_vía"].dropna()
                    if not cambio.empty:
                        info["cambio_vía"] = cambio.iloc[-1]

                # Si hay más de un producto que no sea vacío: error de rotación
                if (
                    train_day["Producto"][
                        np.invert(
                            (train_day["Producto"].eq("Material Vacio"))
                            | (train_day["Producto"].apply(isEmpty))
                        )
                    ]
                    .unique()
                    .shape[0]
                    > 1
                ):
                    info["Movimiento"] = "INCORRECTO"
                    info["EF"] = "INCORRECTO"
                else:
                    info["Movimiento"] = getMovementType(
                        info["T1"], info["T2"], info["Mov_seq"]
                    )

                # Planificación
                if "VíaPlanificada" in train_day.columns:
                    plan_arr = train_day["LlegadaPlanificada"].dropna()
                    info["LlegadaPlanificada"] = (
                        plan_arr.iloc[0] if not plan_arr.empty else pd.NaT
                    )
                    plan_dep = train_day["SalidaPlanificada"].dropna()
                    info["SalidaPlanificada"] = (
                        plan_dep.iloc[-1] if not plan_dep.empty else pd.NaT
                    )
                    info["OcupaciónPlanificada"] = (
                        info["SalidaPlanificada"] - info["LlegadaPlanificada"]
                    )

                # Ocupación
                ini_occ = train_day.loc[
                    train_day["Movimiento"].isin(["LLEGADA", "MANIOBRA", "ALTA"]),
                    "Fecha",
                ]
                if not ini_occ.empty:
                    info["InicioOcupación"] = ini_occ.iloc[0]
                end_occ = train_day.loc[
                    train_day["Movimiento"].isin(["SALIDA", "CAMBIO_VÍA", "MANIOBRA"]),
                    "Fecha",
                ]
                if not end_occ.empty:
                    info["FinOcupación"] = end_occ.iloc[-1]
                info["Ocupación"] = info["FinOcupación"] - info["InicioOcupación"]

                # Anticipación
                # Si es origen tendrá alta y salida
                m0 = pd.NaT
                m1 = pd.NaT
                # TODO: Cómo se tienen en cuenta las rotaciones?
                if info["Movimiento"] == "ORIGEN":
                    m0 = train_day.loc[train_day["Movimiento"] == "ALTA", "Fecha"].iloc[
                        0
                    ]
                    m1 = train_day.loc[
                        train_day["Movimiento"].isin(
                            ["SALIDA", "CAMBIO_VÍA", "MANIOBRA"]
                        ),
                        "Fecha",
                    ].iloc[0]
                    info["AnticipaciónPlataforma"] = m0
                    info["AnticipaciónSalida"] = m1

                else:
                    # Si no, buscamos aproximación y llegada
                    appr = train_day.loc[
                        train_day["Movimiento"].isin(["APROXIMACIÓN", "PREVISIÓN"]),
                        "Fecha",
                    ]
                    arr = train_day.loc[
                        train_day["Movimiento"].isin(["LLEGADA", "MANIOBRA"]),
                        "Fecha",
                    ]
                    if not appr.empty:
                        m0 = appr.iloc[0]
                    if not arr.empty:
                        m1 = arr.iloc[0]
                    info["AnticipaciónAproximación"] = m0
                    info["AnticipaciónLlegada"] = m1
                info["Anticipación"] = m1 - m0

                # # Tiempos totales
                # info["Inicio"] = train_day["Fecha"].iloc[0]
                # info["Fin"] = train_day["Fecha"].iloc[-1]
                # info["TiempoTotal"] = info["Fin"] - info["Inicio"]

                datos.append(info)
        info_estacion = pd.DataFrame(datos)
        return info_estacion

In [25]:
station_historic = procesarSecuenciasEstacion(
    map_estacion_elemento[estacion], mov_sorter
)
# Formateamos fechas/horarios
cols_tiempos = ["Anticipación", "OcupaciónPlanificada", "Ocupación"]
cols_tiempos = [c for c in cols_tiempos if c in station_historic.columns]
station_historic[[f"{c} (segundos)" for c in cols_tiempos]] = station_historic[
    cols_tiempos
].map(lambda x: x.total_seconds() if pd.notna(x) else 0)
station_historic[cols_tiempos] = station_historic[cols_tiempos].map(
    lambda x: formatTimedelta(x.total_seconds()) if pd.notna(x) else x
)


In [26]:
use_df = station_historic.copy()

In [27]:
use_df

,T1,T2,T_seq,Código,ProductoT1,ProductoT2,EF,Vía,TipoVía,LlegadaPlanificada,...,FIN,BAJA,ORIGEN,SALIDA,MANIOBRASALIDA,MANIOBRA,Elemento,Anticipación (segundos),OcupaciónPlanificada (segundos),Ocupación (segundos)
0,10086,10086,10086,05000,NaN,NaN,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,2026-02-20 10:46:25,<NA>,<NA>,2A,0,0,94.0
1,02366,02366,02366,05000,ALVIA,ALVIA,RENFE,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,2026-02-20 20:46:43,<NA>,<NA>,2A,0,0,115.0
2,10086,10086,10086,05000,NaN,NaN,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,2026-02-21 10:40:54,<NA>,<NA>,2A,0,0,96.0
3,10086,10086,10086,05000,NaN,NaN,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,2026-02-22 10:46:06,<NA>,<NA>,2A,0,0,108.0
4,10086,10086,10086,05000,NaN,NaN,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,2026-02-23 10:50:37,<NA>,<NA>,2A,0,0,125.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223,39802,39802,39802,05000,Material Vacio,Material Vacio,RENFE,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,2026-02-25 12:13:57,<NA>,<NA>,6,0,0,0.0
224,08525,08525,08525,05000,AVANT,AVANT,RENFE,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,2026-02-25 12:40:50,<NA>,<NA>,6,0,0,0.0
225,08334,08334,08334,05000,NaN,NaN,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,6,0,0,0.0
226,33575,33575,33575,05000,AVANT,AVANT,RENFE,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,2026-02-25 16:21:12,<NA>,<NA>,6,0,0,1540.0


In [28]:
df_filter = use_df[["T_seq","ProductoT2","Elemento","LLEGADA","SALIDA","Ocupación (segundos)"]].copy()

In [29]:
rename_columns ={
    "T_seq":"NTécnico",
    "ProductoT2":"Producto",
    "Elemento":"Vía",
    "LLEGADA":"LLEGADA",
    "SALIDA":"SALIDA",
    "Ocupación (segundos)":"Ocupación (segundos)"
}

In [30]:
df_filter.rename(columns=rename_columns,inplace=True)

In [31]:
df_filter

,NTécnico,Producto,Vía,LLEGADA,SALIDA,Ocupación (segundos)
0,10086,NaN,2A,2026-02-20 10:44:51,2026-02-20 10:46:25,94.0
1,02366,ALVIA,2A,2026-02-20 20:44:48,2026-02-20 20:46:43,115.0
2,10086,NaN,2A,2026-02-21 10:39:18,2026-02-21 10:40:54,96.0
3,10086,NaN,2A,2026-02-22 10:44:18,2026-02-22 10:46:06,108.0
4,10086,NaN,2A,2026-02-23 10:48:32,2026-02-23 10:50:37,125.0
...,...,...,...,...,...,...
223,39802,Material Vacio,6,<NA>,2026-02-25 12:13:57,0.0
224,08525,AVANT,6,<NA>,2026-02-25 12:40:50,0.0
225,08334,NaN,6,2026-02-25 15:04:02,<NA>,0.0
226,33575,AVANT,6,2026-02-25 15:55:32,2026-02-25 16:21:12,1540.0


In [32]:
lista_con_id = [df for df in df_split if (df["NTécnico"] == "08525").any()]

In [33]:
lista_con_id[5]

,Elemento,Fecha,NTécnico,CTC,Mnemónico,Nombre,Código,Sentido,Movimiento,FuenteMovimiento,FuenteMensaje,Descripción,mov_ord
118,6,2026-02-25 12:13:57,08525,ANT,GRA,Granada,05000,IZQUIERDA,LLEGADA,CTC_MSE,JCTC,"{'train': '08525', 'occupation': True, 'elemen...",4
119,6,2026-02-25 12:40:50,08525,ANT,GRA,Granada,05000,DERECHA,SALIDA,CTC_MSE,JCTC,"{'train': '08525', 'occupation': False, 'eleme...",8


In [34]:
lista_con_id_1 = [df for df in df_split if (df["NTécnico"] == "02197").any()]

In [35]:
lista_con_id_1[0]

,Elemento,Fecha,NTécnico,CTC,Mnemónico,Nombre,Código,Sentido,Movimiento,FuenteMovimiento,FuenteMensaje,Descripción,mov_ord
32,5,2026-02-20 17:52:18,02197,ANT,GRA,Granada,05000,IZQUIERDA,LLEGADA,CTC_MSE,JCTC,"{'train': '02197', 'occupation': True, 'elemen...",4
33,5,2026-02-20 18:31:37,02197,ANT,GRA,Granada,05000,DERECHA,SALIDA,CTC_MSE,JCTC,"{'train': '02197', 'occupation': False, 'eleme...",8


In [36]:
df_filter["LLEGADA"] = pd.to_datetime(df_filter["LLEGADA"], errors="coerce")
df_filter["SALIDA"] = pd.to_datetime(df_filter["SALIDA"], errors="coerce")

df_filter["Fecha"] = (
    df_filter["LLEGADA"]
    .fillna(df_filter["SALIDA"])
    .dt.date
)

In [37]:
df_filter = df_filter[["Fecha","Vía","NTécnico","Producto","LLEGADA","SALIDA","Ocupación (segundos)"]].copy()

In [38]:
fname = Path(r"c:\Users\xiangzhou.zhang\Documents\Data\xPEC\xPEC_20260219033355.xml")

In [39]:
xpec = XPECProcessor()

In [40]:
DF =xpec.readLogFile(fname)

  0%|          | 0/11135 [00:00<?, ?it/s]

In [41]:
df_xpec = pd.DataFrame(DF)

In [42]:
df_xpec["company"].unique()

array(['RF', 'CP', 'IL', 'RI', 'AL', 'FG', 'AD', 'RD', 'CM', 'RM', 'MW',
       'EC', 'CT', 'TT', 'LG', 'CF', 'LC', 'TR', 'GT', 'SF', 'SV'],
      dtype=object)

In [43]:
df_company = df_xpec[["NTécnico","company"]].copy()

In [44]:
df_company["company"].unique()

array(['RF', 'CP', 'IL', 'RI', 'AL', 'FG', 'AD', 'RD', 'CM', 'RM', 'MW',
       'EC', 'CT', 'TT', 'LG', 'CF', 'LC', 'TR', 'GT', 'SF', 'SV'],
      dtype=object)

In [64]:
df_company.drop_duplicates(subset=["NTécnico"],inplace=True)

In [65]:
df_filter = df_filter.merge(
    df_company,
    on="NTécnico",
    how="left"
)

In [66]:
VALOR = {
    "AD": "ADIF",
    "LG":"LOGITREN",
    "LC":"LOW COST",
    "TT":"TRANSERVI",
    "IL":"ILSA",
    "RD":"REDALSA",
    "TR":"T.RAIL",
    "SV":"SNCF VOYAGEURS",
    "FG":"FGC",
    "SF":"HEXAFRET",
    "CT":"CONTINENTAL",
    "CP": "COMBOIOS DE PORTUGAL",
    "RM":"RENFE MERCANCIAS",
    "RI":"OUIGO",
    "CF": "CEFSA",
    "RF":"RENFE",
    "CM":"CAPTRAIN",
    "MW":"MEDWAY",
    "GT":"GO TRANSPORT SERVICIOS S.A.",
    "AL": "ALSA RAIL",
    "FGC":"FGC RAIL",
    "EC":"DB CARGO FRANCE"
    
}

In [67]:
df_company["company"]= df_company["company"].replace(VALOR)

In [68]:
df_company["company"].unique()

array(['RENFE', 'COMBOIOS DE PORTUGAL', 'ILSA', 'OUIGO', 'ALSA RAIL',
       'FGC RAIL', 'ADIF', 'REDALSA', 'CAPTRAIN', 'RENFE MERCANCIAS',
       'MEDWAY', 'DB CARGO FRANCE', 'CONTINENTAL', 'TRANSERVI',
       'LOGITREN', 'CEFSA', 'LOW COST', 'T.RAIL',
       'GO TRANSPORT SERVICIOS S.A.', 'HEXAFRET', 'SNCF VOYAGEURS'],
      dtype=object)

In [69]:
df_filter["company"] = df_filter["company"].replace(VALOR)

In [70]:
df_filter = df_filter[["company","Fecha","Vía","NTécnico","Producto","LLEGADA","SALIDA","Ocupación (segundos)"]].copy()

In [71]:
df_filter.rename(columns ={"company":"Empresa"}, inplace=True)

In [73]:
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\ocupacion_Via_05000.xlsx")

In [ ]:
guardarExcel(df_filter, fname)

: 

In [150]:
use_df["EF"].unique()

array([<NA>, 'RENFE', 'RD'], dtype=object)

In [ ]:
use_df["Fallo"] = False

In [ ]:
fallos = []
for v in use_df["Vía"].unique():
    ex = use_df[use_df["Vía"] == v].sort_values(by="InicioOcupación")
    fail = np.where(
        # fin occ. después que inicio occ. siguiente
        (ex["FinOcupación"] > ex["InicioOcupación"].shift(-1))
        # inicio occ. antes que fin occ. anterior
        | (ex["InicioOcupación"] < ex["FinOcupación"].shift(1))
        # fin occ. antes que inicio occ.
        | (ex["InicioOcupación"] > ex["FinOcupación"])
    )[0]
    fallos.extend(ex.iloc[fail].index.tolist())
use_df.loc[fallos, "Fallo"] = True

In [ ]:
sname = map_cod2name[estacion]
fname = f"{estacion} {map_name2use_name[sname]}"

<h3> saveinfo </h3>

In [ ]:
def getOccTime(
    row: pd.Series,
    min_date: pd.Timestamp,
    max_date: pd.Timestamp,
    hour_period: int = 3,
    margen: int = 20,
):
    """
    Obtener tiempo de ocupación de una vía en una estación por tramos horarios.
    """
    date_range = pd.date_range(
        min_date, max_date, freq=timedelta(hours=hour_period)
    )

    ini = row["InicioOcupación"] - timedelta(minutes=margen)
    fin = row["FinOcupación"] + timedelta(minutes=margen)

    cod = row["Código"]
    via = row["Vía"]
    tipo_via = row.get("TipoVía", None)
    seq = row.get("T_seq", None)

    occ = []
    for i in range(len(date_range) - 1):
        start = date_range[i]
        end = date_range[i + 1]

        if (ini >= start and ini < end) or (ini < start and fin > start):
            duration = (min(fin, end) - max(ini, start)).total_seconds()

            # 👇 ESTRUCTURA FIJA SIEMPRE
            occ.append(
                (
                    cod,
                    via,
                    tipo_via,
                    start,
                    duration,
                    seq,
                )
            )

    return occ


In [ ]:
def getSaturation(
        df: pd.DataFrame,
        min_date: pd.Timestamp,
        max_date: pd.Timestamp,
        hour_period: int = 3,
        margen: int = 20,
        modo: str = "ocupado",
    ):
        """
        Obtener la saturación de vías de todas las estaciones por tramos horarios (periodos de `seconds_period` segundos).
        modo: {"ocupado", "libre"}
        """
        if df is None or df.empty:
            return None

        df_aux = df.copy()
        # Rellenamos valores vacíos
        df_aux.loc[df_aux["InicioOcupación"].isna(), "InicioOcupación"] = df_aux.loc[
            df_aux["InicioOcupación"].isna(), "FinOcupación"
        ]
        df_aux.loc[df_aux["FinOcupación"].isna(), "FinOcupación"] = df_aux.loc[
            df_aux["FinOcupación"].isna(), "InicioOcupación"
        ]
        df_aux = df_aux.dropna(subset=["InicioOcupación", "FinOcupación"], how="all")

        if modo == "ocupado":
            saturation = pd.DataFrame(
                df_aux.apply(
                    getOccTime,
                    min_date=min_date,
                    max_date=max_date,
                    hour_period=hour_period,
                    margen=margen,
                    axis=1,
                )
                .explode()
                .dropna()
                .tolist(),
                columns=["Código", "Vía", "TipoVía", "Fecha", "Ocupación", "T_seq"],
            )
            saturation = (
                saturation.groupby(
                    by=["Código", "Vía", "TipoVía", "Fecha"], dropna=False
                )
                .agg({"Ocupación": "sum", "T_seq": lambda x: "→".join(x)})
                .reset_index()
            )
            saturation["Trenes"] = saturation["T_seq"].apply(
                lambda x: list(set(x.split("→")))
            )
            saturation["FinOcupación"] = saturation["Fecha"] + saturation[
                "Ocupación"
            ].apply(lambda x: timedelta(seconds=x))
            total_secs = hour_period * 3600
            saturation["Occ"] = saturation["Ocupación"] / total_secs
            saturation["Ocupación (%)"] = saturation["Occ"].apply(
                lambda x: f"{x*100:.3f}%"
            )
        elif modo == "libre":
            print("libre")
            saturation = pd.DataFrame(
                df_aux.apply(
                    getOccTime,
                    min_date=min_date,
                    max_date=max_date,
                    hour_period=hour_period,
                    margen=margen,
                    axis=1,
                )
                .explode()
                .dropna()
                .tolist(),
                columns=["Código", "Vía","TipoVía","Fecha", "Libre","T_seq"],
            )
            saturation = (
                saturation.groupby(
                    by=["Código", "Vía", "Fecha"], dropna=False
                )
                .agg({"Libre": "sum"})
                .reset_index()
            )
            saturation["FinLibre"] = saturation.apply(
                lambda x: x["Fecha"] + timedelta(seconds=x["Libre"]), axis=1
            )
            total_secs = hour_period * 3600
            saturation["Lib"] = saturation["Libre"] / total_secs
            saturation["Libre (%)"] = saturation["Lib"].apply(lambda x: f"{x*100:.3f}%")
        return saturation

In [ ]:
def getTramosLibres( df: pd.DataFrame, margen: int = 10, t_min: int = 5):
    """
    Genera un dataframe de tramos libres a partir de un dataframe de ocupación.
    Parámetros:
    -----------
    df: pd.DataFrame
        Tabla de ocupaciones con, al menos:
        - "InicioOcupación"
        - "FinOcupación"
        - "Código"
        - "Vía"
        - "TipoVía"

    margen: int
        Tiempo de seguridad mínimo (en minutos) entre ocupaciones.
    t_min: int
        Duración mínima de ocupación (en minutos).
    """
    df_aux = df.copy()
    # Rellenamos valores vacíos
    df_aux.loc[df_aux["InicioOcupación"].isna(), "InicioOcupación"] = df_aux.loc[
        df_aux["InicioOcupación"].isna(), "FinOcupación"
    ]
    df_aux.loc[df_aux["FinOcupación"].isna(), "FinOcupación"] = df_aux.loc[
        df_aux["FinOcupación"].isna(), "InicioOcupación"
    ]

    # Se incluyen las fechas de las que se dispone en el dataframe
    if df_aux[["InicioOcupación", "FinOcupación"]].dropna().empty:
        return pd.DataFrame(
            columns=[
                "Código",
                "Elemento",
                # "TipoVía",
                "InicioLibre",
                "FinLibre",
                "Libre (segundos)",
                "Libre",
            ]
        )
    min_date = df_aux[["InicioOcupación", "FinOcupación"]].dropna().values.min()
    max_date = df_aux[["InicioOcupación", "FinOcupación"]].dropna().values.max()
    data_free_time = []
    for cod in df_aux["Código"].unique():
        for via in df_aux.loc[df_aux["Código"] == cod, "Elemento"].unique():

            df_via = (
                df_aux[(df_aux["Código"] == cod) & (df_aux["Elemento"] == via)]
                .sort_values("InicioOcupación")
                .reset_index(drop=True)
            )

            # --- Intervalo libre ANTES de la primera ocupación ---
            ini_libre = min_date
            fin_libre = df_via.loc[0, "InicioOcupación"] - timedelta(minutes=margen)

            if fin_libre > ini_libre:
                data_free_time.append([
                    cod, via, ini_libre, fin_libre,
                    (fin_libre - ini_libre).total_seconds()
                ])

            # --- Intervalos libres ENTRE ocupaciones ---
            for i in range(len(df_via) - 1):
                ini_libre = df_via.loc[i, "FinOcupación"] + timedelta(minutes=margen)
                fin_libre = df_via.loc[i + 1, "InicioOcupación"] - timedelta(minutes=margen)

                if fin_libre > ini_libre:
                    data_free_time.append([
                        cod, via, ini_libre, fin_libre,
                        (fin_libre - ini_libre).total_seconds()
                    ])

            # --- Intervalo libre DESPUÉS de la última ocupación ---
            ini_libre = df_via.loc[len(df_via) - 1, "FinOcupación"] + timedelta(minutes=margen)
            fin_libre = max_date

            if fin_libre > ini_libre:
                data_free_time.append([
                    cod, via, ini_libre, fin_libre,
                    (fin_libre - ini_libre).total_seconds()
                ])


    map_tipo_via_tiempo = {"AV": 45, "RC": 25}
    df_free = pd.DataFrame(
        data_free_time,
        columns=[
            "Código",
            "Elemento",
            "InicioLibre",
            "FinLibre",
            "Libre (segundos)",
        ],
    ).dropna()

    df_free["Libre"] = df_free["Libre (segundos)"].apply(formatTimedelta)
    # df_free = df_free[
    #     df_free[["Libre (segundos)", "InicioLibre", "FinLibre"]].apply(
    #         lambda x: (
    #             x["Libre (segundos)"]
    #             >= map_tipo_via_tiempo.get(x["TipoVía"], 0) * 60
    #         )
    #         # & (x["TipoVía"] in map_tipo_via_tiempo.keys())
    #         & (x["FinLibre"] > x["InicioLibre"]),
    #         axis=1,
    #     )
    # ]
    # df_free = df_free[
    #     (df_free["Libre (segundos)"] >= t_min * 60)
    #     & (df_free["FinLibre"] > df_free["InicioLibre"])
    # ]
    df_free[["HoraInicioLibre", "HoraFinLibre"]] = df_free[
        ["InicioLibre", "FinLibre"]
    ].map(lambda x: x.strftime("%H:%M:%S") if x and pd.notna(x) else "")
    return df_free

<h3> Saturación </h3>

In [ ]:
def setHoverInfo(df: pd.DataFrame, hover_cols: list[str]):
    hover_cols = [c for c in hover_cols if c in df.columns]
    # Máxima longitud de nombre de columnas
    c_len = max([len(c) for c in hover_cols]) + 2
    # Fijamos la longitud máxima del valor, si se supera, hay un salto de línea
    fd_len = min(
        df[hover_cols]
        .fillna("")
        .map(lambda x: len(f"{x}"), na_action="ignore")
        .max()
        .max(),
        25,
    )


In [ ]:
def addRectTrace(
    inicio,
    fin,
    y_inicio,
    y_fin,
    dy,
    mode,
    color,
    name,
    hover_info,
    showlegend: bool = True,
    opacity: float = 1,
    width: float = 0,
    dash=None,
):
    if mode == "markers":
        x = (inicio or fin,)
        y = (y_inicio or y_fin,)
        fill = None
    else:
        x = (inicio, inicio, fin, fin, inicio)
        y = (y_inicio + dy, y_inicio - dy, y_fin - dy, y_fin + dy, y_inicio + dy)
        fill = "toself"

    trace = go.Scatter(
        x=x,
        y=y,
        mode=mode,
        line=dict(color=color, width=width, dash=dash),
        visible=True,
        fill=fill,
        fillcolor=color,
        hoverinfo="text",
        hovertext=hover_info,
        # hoveron="points+fills",
        hoveron="points",
        textfont=dict(family="calibri", size=18),
        name=name,
        showlegend=showlegend,
        legendgroup=name,
        opacity=opacity,
    )
    return trace
map_EF_color = {
    "RENFE": "#830065",
    "IRYO": "#DA291C",
    "OUIGO": "#0096CA",
    "INCORRECTO": "khaki",
    "Otro": "mediumblue",
}
color_sorter = {
    c: i
    for i, c in enumerate(
        [
            "#830065",  # pantone 2425c renfe
            "#DA291C",  # pantone 485c iryo
            "#0096CA",  # azul ouigo
            "silver",
            "mediumblue",
            "green",
            "yellow",
            "khaki",
            "orange",
            "red",
            "darkred",
            "black",
        ]
    )
}
map_color_EF = {c: ef for ef, c in map_EF_color.items()}
def set_color_ocupacion(s, criterio="EF"):
    """
    Establece el color de la ocupación en función de un criterio.
    criterio: {"EF", "tiempo"}
    """
    # if criterio == "EF":
    #     if regex.search(
    #         r"(ALVIA|AVANT|AVE|CERCANIAS|INTERCITY|MD|REGIONAL EXPRES|TALGO|AVLO)", s
    #     ):
    #         return "#830065"
    #     if regex.search(r"(IRYO)", s):
    #         return "#DA291C"
    #     elif regex.search(r"(OUIGO)", s):
    #         return "#0096CA"
    #     else:
    #         return "mediumblue"
    if criterio == "EF":
        return map_EF_color.get(s, "mediumblue")

    elif criterio == "tiempo":
        if s < 5:
            return "mediumblue"
        elif s <= 120:
            return "green"
        elif s < 180:
            return "orange"
        elif s <= 300:
            return "red"
        elif s > 300:
            return "darkred"
        else:
            return "black"
map_color_ocupacion = {
    "mediumblue": "<5 segundos",
    "green": "<120 segundos",
    # "yellow": "<180 segundos",
    "orange": "<180 segundos",
    "red": "<300 segundos",
    "darkred": ">300 segundos",
    "black": "Error",
}
def set_name(
    color: str = None,
    shape: str = None,
    map_color: dict = None,
    map_shape: dict = None,
):
    name_parts = []
    if map_color:
        name_parts.append(map_color[color])
    if map_shape:
        name_parts.append(map_shape[shape])
    return ", ".join(name_parts)

def incluirOcupaciones(df: pd.DataFrame, platform_map: dict, criterio: str = "EF"):
    if df.empty:
        return []

    df_rep = (
        df.copy()
        # .sort_values(by=["LlegadaPlanificada"])
        .reset_index(drop=True)
    )

    # Definimos el color
    if criterio == "EF":
        map_color = map_color_EF
        if "EF" not in df_rep.columns:
            df_rep["EF"] = df_rep["Producto"].apply(setEF)
        df_rep["color"] = df_rep["EF"].apply(
            lambda x: set_color_ocupacion(x, criterio=criterio)
        )
        # Si el movimiento es incorrecto, independientemente de EF, lo marcamos como tal
        df_rep.loc[df_rep["Movimiento"] == "INCORRECTO", "color"] = set_color_ocupacion(
            "INCORRECTO", criterio=criterio
        )
    elif criterio == "tiempo":
        map_color = map_color_ocupacion
        df_rep["color"] = df_rep["OcupaciOcupación (segundos)"].apply(
            lambda x: (
                set_color_ocupacion(x, criterio=criterio)
                if pd.notna(x)
                else set_color_ocupacion(0, criterio=criterio)
            )
        )

    # Info para mostrar
    df_rep[["HoraInicioOcupación", "HoraFinOcupación"]] = df_rep[
        ["InicioOcupación", "FinOcupación"]
    ].map(lambda x: x.strftime("%H:%M:%S") if pd.notna(x) else "")
    hover_cols = [
        "NTécnico",
        "Mov_seq",
        "EF",
        "Producto",
        "Movimiento",
        "HoraInicioOcupación",
        "HoraFinOcupación",
        "Ocupación",
        "Vía",
        # "TipoVía",
    ]
    df_rep["hover_info"] = setHoverInfo(df_rep, hover_cols)

    # Ordenamos las vías
    df_rep["Vía_order"] = df_rep["Vía"].apply(platform_map.get)

    traces = []
    used_names = []
    for c in sorted(df_rep["color"].unique(), key=color_sorter.get):
        df_aux = df_rep[df_rep["color"] == c]
        name = set_name(color=c, map_color=map_color)

        trace = addRectTrace(
            inicio=(None,),
            fin=(None,),
            y_inicio=0,
            y_fin=0,
            dy=0,
            mode="lines",
            color=c,
            name=name,
            hover_info=None,
            showlegend=True,
            opacity=1,
            width=0,
        )
        if name not in used_names:
            used_names.append(name)
        traces.append(trace)

        for _, row in df_aux.sort_values(by=["InicioOcupación"]).iterrows():
            if row["Vía"] not in platform_map:
                continue
            if row["Ocupación (segundos)"] < 5:
                inicio = row["InicioOcupación"] or row["FinOcupación"]
                fin = row["InicioOcupación"] or row["FinOcupación"]
                opacity = 1
                dy = 0
                mode = "markers"
                width = 0
            else:
                inicio = row["InicioOcupación"]
                fin = row["FinOcupación"]
                dy = 0.15
                opacity = 1
                mode = "lines"
                width = 1

            trace = addRectTrace(
                inicio=inicio,
                fin=fin,
                y_inicio=row["Vía_order"],
                y_fin=row["Vía_order"],
                dy=dy,
                mode=mode,
                color=c,
                name=name,
                hover_info=row["hover_info"],
                showlegend=True if name not in used_names else False,
                opacity=opacity,
                width=width,
            )
            traces.append(trace)
            if name not in used_names:
                used_names.append(name)
    return traces



In [ ]:
def incluirLibre(df: pd.DataFrame, platform_map: dict, margen=10, t_min=10):
    """
    margen: int
        Tiempo de seguridad mínimo (en minutos) entre ocupaciones.
    t_min: int
        Duración mínima de ocupación (en minutos).
    """
    if df.empty:
        return []

    df_rep = df.copy()

    # Info para mostrar
    hover_cols_free = [
        "Vía",
        "TipoVía",
        "HoraInicioLibre",
        "HoraFinLibre",
    ]
    df_rep["hover_info"] = setHoverInfo(df_rep, hover_cols_free)

    # Ordenamos las vías
    df_rep.rename(columns={"Elemento":"Vía"}, inplace=True)
    df_rep["Vía_order"] = df_rep["Vía"].apply(platform_map.get)

    opacity = 0.66
    width = 1
    mode = "lines"
    name = f"Libre (>{t_min} min)"

    traces = []
    for i, row in df_rep.iterrows():
        if row["Vía"] not in platform_map:
            continue
        trace = addRectTrace(
            inicio=row["InicioLibre"],
            fin=row["FinLibre"],
            y_inicio=row["Vía_order"],
            y_fin=row["Vía_order"],
            dy=0.35,
            mode=mode,
            color="silver",
            name=name,
            hover_info=row["hover_info"],
            showlegend=True if not i else False,
            opacity=opacity,
            width=width,
        )
        traces.append(trace)
    return traces

In [ ]:
from src.visualizacion.utils import BGCOLOR, getSortedPlatforms, setHoverInfo, setLayout
def visualizacionOcupacionVia(
    df: pd.DataFrame,
    title: str = "",
    df_free: pd.DataFrame = None,
    margenes: pd.DataFrame = None,
    show_plan: bool = True,
    show_fail: bool = True,
):
    # En principio el criterio de colores es la empresa ferroviaria
    criterio = "EF"

    if df is None or df.empty:
        return

    traces = []
    use_df = df.copy().replace([pd.NA], [None])
    # Ordenamos las vías por número
    platform_map = getSortedPlatforms(
        use_df["Vía"].dropna().unique(), use_df["Código"].iloc[0]
    )
    # Pintamos tiempos libres
    if df_free is not None and not df_free.empty:
        traces.extend(incluirLibre(df_free, platform_map, margen=10, t_min=40))
    # # Pintamos los fallos
    # if show_fail:
    #     traces.extend(incluirFallos(use_df[use_df["Fallo"]], platform_map))
    # # Incluir planificación
    # if show_plan and "Ocupación planificada (segundos)" in use_df.columns:
    #     traces.extend(
    #         incluirPlanificacion(use_df, platform_map=platform_map, criterio=criterio)
    #     )
    # # Incluir planificación
    # if margenes is not None and not margenes.empty:
    #     traces.extend(incluirMargenes(margenes, platform_map=platform_map, margen=10))
    # # Pintamos los cambios de vía
    # if criterio == "EF":
    #     traces.extend(
    #         incluirCambioVia(
    #             use_df.dropna(subset="cambio_vía"), platform_map=platform_map
    #         )
    #     )
    # Pintamos las ocupaciones normales
    traces.extend(
        incluirOcupaciones(
            use_df[~use_df["Fallo"]], platform_map=platform_map, criterio=criterio
        )
    )

    # Creamos el layout
    min_date = pd.to_datetime(
        (pd.to_datetime(use_df["FinOcupación"]).min() - timedelta(hours=0.5)).strftime(
            "%Y-%m-%d %H"
        )
    ) - timedelta(hours=1)
    max_date = pd.to_datetime(
        (pd.to_datetime(use_df["FinOcupación"]).max() + timedelta(hours=0.5)).strftime(
            "%Y-%m-%d %H"
        )
    ) + timedelta(hours=1)
    layout = setLayout("togglegroup", title, platform_map, x_range=(min_date, max_date))
    # layout = None

    fig = go.Figure(data=traces, layout=layout)

    return fig

In [ ]:
df = use_df.copy()

In [ ]:
df.columns

In [ ]:
df["Vía"] = df["Elemento"]

In [ ]:

save_cols = (
    ["Código", "Estación", "Vía", "TipoVía"]
    + ["T1", "T2", "T_seq"]
    + ["ProductoT1", "ProductoT2", "Producto"]
    + ["Movimiento", "Mov_seq", "full_seq", "cambio_vía"]
    + ["ALTA", "APROXIMACIÓN", "MANIOBRALLEGADA", "LLEGADA", "SALIDA"]
    + ["MANIOBRASALIDA", "FIN", "BAJA", "MANIOBRA", "Anticipación"]
    + ["InicioOcupación", "FinOcupación", "Ocupación (segundos)", "Ocupación"]
    + [
        "VíaPlanificada",
        "LlegadaPlanificada",
        "SalidaPlanificada",
        "OcupaciónPlanificada (segundos)",
        "OcupaciónPlanificada",
    ]
    + ["RotaciónValidada", "Fallo"]
)
df["Fallo"] = df["Fallo"].apply(lambda x: True if x and pd.notna(x) else False)

# Buscamos fechas límites para rangos
min_date = pd.to_datetime(
    (pd.to_datetime(df["FinOcupación"]).min() - timedelta(hours=0.5)).strftime(
        "%Y-%m-%d %H"
    )
) - timedelta(hours=1)
max_date = pd.to_datetime(
    (pd.to_datetime(df["FinOcupación"]).max() + timedelta(hours=0.5)).strftime(
        "%Y-%m-%d %H"
    )
) + timedelta(hours=1)

# Obtenemos la saturación de las vías
saturation_1h = getSaturation(
    df, min_date, max_date, hour_period=1, margen=0
)
saturation_3h = getSaturation(
    df, min_date, max_date, hour_period=3, margen=0
)

# Generamos visualización
margen = 10
t_min = 5
df_free = (
    getTramosLibres(df, margen=margen, t_min=t_min)
    .sort_values(by=["InicioLibre"])
    .reset_index(drop=True)
)
# df_free.rename(columns={"Elemento": "Vía"}, inplace=True)
# if df_free is not None:
#     print("A")
#     saturation_libre_1h = getSaturation(
#         df_free.rename(
#             columns={
#                 "InicioLibre": "InicioOcupación",
#                 "FinLibre": "FinOcupación",
#             }
#         ),
#         min_date,
#         max_date,
#         hour_period=1,
#         margen=0,
#         modo="libre",
#     )
#     saturation_libre_3h = getSaturation(
#         df_free.rename(
#             columns={
#                 "InicioLibre": "InicioOcupación",
#                 "FinLibre": "FinOcupación",
#             }
#         ),
#         min_date,
#         max_date,
#         hour_period=3,
#         margen=0,
#         modo="libre",
#     )
# else:
#     saturation_libre_1h = None
#     saturation_libre_3h = None
# fig = visualizacionOcupacionVia(
#     df,
#     title=f"Ocupación de vías en <b>{"nombre_estacion"}</b>",
#     df_free=df_free,
#     show_fail=True,
#     show_plan=True,
# )
# fig_sat_1h = visualizacionSaturacionVia(
#     saturation_1h,
#     saturation_libre_1h,
#     title=f"Saturación de vías en <b>{nombre_estacion}</b> (periodos 1h)",
# )
# fig_sat_3h = visualizacionSaturacionVia(
#     saturation_3h,
#     saturation_libre_3h,
#     title=f"Saturación de vías en <b>{nombre_estacion}</b> (periodos 3h)",
# )
# fig_ant = visualizeAnticipacionVia(
#     df, title=f"Anticipación de vías en <b>{nombre_estacion}</b>"
# )

# # # Guardamos los datos
# # save_loc = Path(save_dir).joinpath(fname)
# # save_loc.mkdir(parents=True, exist_ok=True)

# # if not df.empty:
# #     guardarExcel(
# #         df[[c for c in save_cols if c in df.columns]],
# #         save_loc.joinpath(f"historico {days}.xlsx"),
# #         append_sheet=False,
# #     )
# # if fig_occ:
# #     fig_occ.write_html(save_loc.joinpath(f"ocupación {days}.html"))
# #     # fig_occ.write_image(save_loc.joinpath(f"ocupación {days}.png"))
# # if fig_sat_1h:
# #     fig_sat_1h.write_html(save_loc.joinpath(f"saturación_1h {days}.html"))
# # if fig_sat_3h:
# #     fig_sat_3h.write_html(save_loc.joinpath(f"saturación_3h {days}.html"))
# # if fig_ant:
# #     fig_ant.write_html(save_loc.joinpath(f"anticipación {days}.html"))

In [ ]:
df_free

In [ ]:
fig.show()

In [ ]:
margen = 10
t_min = 5
df_free = (
    getTramosLibres(df, margen=margen, t_min=t_min)
    .sort_values(by=["InicioLibre"])
    .reset_index(drop=True)
)

In [ ]:
df_aux = df.copy()
# Rellenamos valores vacíos
df_aux.loc[df_aux["InicioOcupación"].isna(), "InicioOcupación"] = df_aux.loc[
    df_aux["InicioOcupación"].isna(), "FinOcupación"
]
df_aux.loc[df_aux["FinOcupación"].isna(), "FinOcupación"] = df_aux.loc[
    df_aux["FinOcupación"].isna(), "InicioOcupación"
]

In [ ]:
if df_aux[["InicioOcupación", "FinOcupación"]].dropna().empty:
    result =  pd.DataFrame(
        columns=[
            "Código",
            "Elemento",
            # "TipoVía",
            "InicioLibre",
        "FinLibre",
            "Libre (segundos)",
            "Libre",
        ]
        )

In [ ]:
df["Código"].unique()

In [ ]:
min_date = df_aux[["InicioOcupación", "FinOcupación"]].dropna().values.min()
max_date = df_aux[["InicioOcupación", "FinOcupación"]].dropna().values.max()
data_free_time = []
for cod in df_aux["Código"].unique():
    for via in df_aux.loc[df_aux["Código"] == cod, "Elemento"].unique():

        df_via = (
            df_aux[(df_aux["Código"] == cod) & (df_aux["Elemento"] == via)]
            .sort_values("InicioOcupación")
            .reset_index(drop=True)
        )

        # --- Intervalo libre ANTES de la primera ocupación ---
        ini_libre = min_date
        fin_libre = df_via.loc[0, "InicioOcupación"] - timedelta(minutes=margen)

        if fin_libre > ini_libre:
            data_free_time.append([
                cod, via, ini_libre, fin_libre,
                (fin_libre - ini_libre).total_seconds()
            ])

        # --- Intervalos libres ENTRE ocupaciones ---
        for i in range(len(df_via) - 1):
            ini_libre = df_via.loc[i, "FinOcupación"] + timedelta(minutes=margen)
            fin_libre = df_via.loc[i + 1, "InicioOcupación"] - timedelta(minutes=margen)

            if fin_libre > ini_libre:
                data_free_time.append([
                    cod, via, ini_libre, fin_libre,
                    (fin_libre - ini_libre).total_seconds()
                ])

        # --- Intervalo libre DESPUÉS de la última ocupación ---
        ini_libre = df_via.loc[len(df_via) - 1, "FinOcupación"] + timedelta(minutes=margen)
        fin_libre = max_date

        if fin_libre > ini_libre:
            data_free_time.append([
                cod, via, ini_libre, fin_libre,
                (fin_libre - ini_libre).total_seconds()
            ])


In [ ]:
df_aux.columns

In [ ]:
if not df_split:
    display("no existe trenes")
for df_t in df_split:
    if df_t.empty:
        display("esta vacio")
    df_t = pd.merge(
        producto,
        df_t,
        on=["NTécnico"],
        how = "right"
    )
    df_t.drop_duplicates(subset=["NTécnico", "Fecha", "Elemento"],inplace=True)
    cols = list(df_t.columns)
    if "Fecha" in cols:
        cols.insert(0, cols.pop(cols.index("Fecha")))
        df_t = df_t[cols]
    df_t["tdiff"] = df_t["Fecha"].diff().dt.total_seconds()
    df_t["mdiff"] = df_t["mov_ord"].diff()
    df_t = df_t.reset_index(drop=True).reset_index()
    df_t["prev_index"] = df_t["index"].shift(fill_value=-1)
    for i, row in df_t[1:].iterrows():
        if (row["tdiff"] < 5) & (row["mdiff"] < 0):
            row = row.copy()
            aux_row = df_t.loc[row["prev_index"]].copy()
            mdate = [row["Fecha"], aux_row["Fecha"]]
            aux_row["Fecha"] = max(mdate)
            row["Fecha"] = min(mdate)
            df_t.loc[row["prev_index"]] = row
            df_t.loc[i] = aux_row
    df_t_split = np.split(
        df_t,
        np.where((~df_t["Elemento"].eq(df_t["Elemento"].shift())))[0][1:],
            )
    apariciones = len(df_t_split)
    for n, train_day in enumerate(df_t_split, 1):
        # Si no hay ninguna vía pasamos
        if (train_day["Elemento"].isna().all()) or (
            train_day["Movimiento"]
            .apply(
                lambda x: x
                in [
                    "APROXIMACIÓN",
                    "PREVISIÓN",
                    "ALTA",
                ]
            )
            .all()
        ):
            continue
        # Si aparece en otra vía, creamos una "salida" provisional a la hora que aparece en la otra
        if n < apariciones and not train_day.iloc[-1]["Movimiento"] == "SALIDA":
            # Comprobamos que la siguiente aparición no sea aproximación
            next_aparicion = df_t_split[n].dropna(subset=["Elemento"])
            next_aparicion = next_aparicion[
                np.invert(
                    next_aparicion["Movimiento"].apply(
                        lambda x: x
                        in [
                            "APROXIMACIÓN",
                            "PREVISIÓN",
                            # "ALTA",
                        ]
                    )
                )
            ]
            if not next_aparicion.empty:
                new_row = train_day.iloc[-1:].copy()
                next_aparicion = next_aparicion.iloc[0]
                # new_row["Fecha"] = next_aparicion["Fecha"]
                new_row["Movimiento"] = "CAMBIO_VÍA"
                new_row["cambio_vía"] = [
                    {
                        "Fecha": next_aparicion["Fecha"],
                        "Elemento": next_aparicion["Elemento"],
                    }
                ]
                # # Asignamos la fecha de la siguiente aparición
                # new_fecha = next_aparicion.iloc[0]["Fecha"]
                # new_row["Fecha"] = new_fecha
                train_day = pd.concat([train_day, new_row])
        via = train_day["Elemento"].dropna().iloc[0]
        map_estacion_elemento[estacion][via].append(train_day)


In [ ]:
data_free_time = []
for cod in df_aux["Código"].unique():
    for via in df_aux.loc[df_aux["Código"] == cod, "Elemento"].unique():

        df_via = (
            df_aux[(df_aux["Código"] == cod) & (df_aux["Elemento"] == via)]
            .sort_values("InicioOcupación")
            .reset_index(drop=True)
        )

        # --- Intervalo libre ANTES de la primera ocupación ---
        ini_libre = min_date
        fin_libre = df_via.loc[0, "InicioOcupación"] - timedelta(minutes=margen)

        if fin_libre > ini_libre:
            data_free_time.append([
                cod, via, ini_libre, fin_libre,
                (fin_libre - ini_libre).total_seconds()
            ])

        # --- Intervalos libres ENTRE ocupaciones ---
        for i in range(len(df_via) - 1):
            ini_libre = df_via.loc[i, "FinOcupación"] + timedelta(minutes=margen)
            fin_libre = df_via.loc[i + 1, "InicioOcupación"] - timedelta(minutes=margen)

            if fin_libre > ini_libre:
                data_free_time.append([
                    cod, via, ini_libre, fin_libre,
                    (fin_libre - ini_libre).total_seconds()
                ])

        # --- Intervalo libre DESPUÉS de la última ocupación ---
        ini_libre = df_via.loc[len(df_via) - 1, "FinOcupación"] + timedelta(minutes=margen)
        fin_libre = max_date

        if fin_libre > ini_libre:
            data_free_time.append([
                cod, via, ini_libre, fin_libre,
                (fin_libre - ini_libre).total_seconds()
            ])

In [ ]:
data_free_time

In [ ]:
df_via

In [ ]:
if not df_split:
    display("no existe trenes")
for df_t in df_split:
    if df_t.empty:
        display("esta vacio")
    df_t = pd.merge(
        producto,
        df_t,
        on=["NTécnico"],
        how = "right"
    )
    df_t.drop_duplicates(subset=["NTécnico", "Fecha", "Elemento"],inplace=True)
    cols = list(df_t.columns)
    if "Fecha" in cols:
        cols.insert(0, cols.pop(cols.index("Fecha")))
        df_t = df_t[cols]
    df_t["tdiff"] = df_t["Fecha"].diff().dt.total_seconds()
    df_t["mdiff"] = df_t["mov_ord"].diff()
    df_t = df_t.reset_index(drop=True).reset_index()
    df_t["prev_index"] = df_t["index"].shift(fill_value=-1)
    for i, row in df_t[1:].iterrows():
        if (row["tdiff"] < 5) & (row["mdiff"] < 0):
            row = row.copy()
            aux_row = df_t.loc[row["prev_index"]].copy()
            mdate = [row["Fecha"], aux_row["Fecha"]]
            aux_row["Fecha"] = max(mdate)
            row["Fecha"] = min(mdate)
            df_t.loc[row["prev_index"]] = row
            df_t.loc[i] = aux_row
    df_t_split = np.split(
        df_t,
        np.where((~df_t["Elemento"].eq(df_t["Elemento"].shift())))[0][1:],
            )
    apariciones = len(df_t_split)
    for n, train_day in enumerate(df_t_split, 1):
        # Si no hay ninguna vía pasamos
        if (train_day["Elemento"].isna().all()) or (
            train_day["Movimiento"]
            .apply(
                lambda x: x
                in [
                    "APROXIMACIÓN",
                    "PREVISIÓN",
                    "ALTA",
                ]
            )
            .all()
        ):
            continue
        # Si aparece en otra vía, creamos una "salida" provisional a la hora que aparece en la otra
        if n < apariciones and not train_day.iloc[-1]["Movimiento"] == "SALIDA":
            # Comprobamos que la siguiente aparición no sea aproximación
            next_aparicion = df_t_split[n].dropna(subset=["Elemento"])
            next_aparicion = next_aparicion[
                np.invert(
                    next_aparicion["Movimiento"].apply(
                        lambda x: x
                        in [
                            "APROXIMACIÓN",
                            "PREVISIÓN",
                            # "ALTA",
                        ]
                    )
                )
            ]
            if not next_aparicion.empty:
                new_row = train_day.iloc[-1:].copy()
                next_aparicion = next_aparicion.iloc[0]
                # new_row["Fecha"] = next_aparicion["Fecha"]
                new_row["Movimiento"] = "CAMBIO_VÍA"
                new_row["cambio_vía"] = [
                    {
                        "Fecha": next_aparicion["Fecha"],
                        "Elemento": next_aparicion["Elemento"],
                    }
                ]
                # # Asignamos la fecha de la siguiente aparición
                # new_fecha = next_aparicion.iloc[0]["Fecha"]
                # new_row["Fecha"] = new_fecha
                train_day = pd.concat([train_day, new_row])
        via = train_day["Elemento"].dropna().iloc[0]
        map_estacion_elemento[estacion][via].append(train_day)


In [ ]:
df_free

<h1> GOV JTCT </H1>

In [ ]:
topo = getEstacionamientos(["18000"])

In [ ]:
topo = topo[["VíaTécnica","Vía"]]

In [ ]:
topo = topo[["VíaTécnica","Vía"]]
topo.rename(columns={"VíaTécnica":"Elemento"},inplace=True)

In [ ]:
df_name_cod = pd.read_excel("data/nombre-codigo.xlsx")
map_cod2name = dict(df_name_cod[["código", "nombre"]].values)
map_use_name2name = dict(df_name_cod[["use_name", "nombre"]].values)
map_name2use_name = dict(df_name_cod[["nombre", "use_name"]].values)


In [ ]:
df = historico_pro.copy()

In [ ]:
topo

In [ ]:
from typing import Union


def filterTrains(
    df: pd.DataFrame,
    day: Union[str, pd.Timestamp] = None,
    platform: str = None,
    station: str = None,
):
    """
    Devuelve un dataframe con los datos de un día en una vía en una estación
    """

    filt_df = df.copy()

    if day is not None:
        day = pd.to_datetime(day)
        filt_df = filt_df[filt_df["Fecha"].dt.date == day.date()]

    if station is not None:
        filt_df = filt_df[filt_df["Código"] == station]

    if platform is not None:
        trenes = filt_df[filt_df["Vía"] == platform]["NTécnico"].unique()
        filt_df = filt_df[
            (filt_df["NTécnico"].isin(trenes))
            & ((filt_df["Vía"] == platform) | (filt_df["Vía"].isna()))
        ]

    return filt_df

In [ ]:
from typing import Union
import pandas as pd

def splitTrainsByDate(
        df: pd.DataFrame,
        day: Union[str, pd.Timestamp] = None,
        platform: str = None,
        station: str = None,
        hour_diff: int = 1,
        filter_mov: bool = False,
    ):
        """
        Separa los datos de un tren en una vía un día concreto
        hour_diff hace que los trenes del mismo número separados por más de estas horas se cuenten por separado
        """
        filt_df = filterTrains(
            df=df,
            day=day,
            platform=platform,
            station=station,
        )
        if filt_df.empty:
            return None, None

        # Agrupar trenes por número técnico y fecha.
        filt_df = filt_df.sort_values(by=["NTécnico", "Fecha", "mov_ord"]).reset_index(
            drop=True
        )
        filt_df["tdiff"] = filt_df["Fecha"].apply(pd.to_datetime, dayfirst=True).diff()
        # No incluimos la columna tdiff
        t_cols = filt_df.columns[:-1]
        df_split = np.split(
            filt_df[t_cols],
            np.where(
                (~(filt_df["tdiff"] < timedelta(hours=hour_diff)))
                | (~filt_df["NTécnico"].eq(filt_df["NTécnico"].shift()))
            )[0][1:],
        )

        # Como las finalizaciones no tienen vía, nos aseguramos de que no se asigna una vía
        # a un fin si no hay llegada.
        col_fecha = np.where(t_cols == "Fecha")[0][0]
        if not filter_mov:
            return sorted(df_split, key=lambda x: x.iloc[-1]["NTécnico"]), t_cols
        filt_split = []
        for sp in df_split:
            aux_df = pd.DataFrame(sp, columns=t_cols)
            movs = "_".join(aux_df["Movimiento"])
            if (("FIN" in movs) or ("BAJA" in movs)) and not ("LLEGADA" in movs):
                # wrong_fin = np.in1d(sp[:, 1], ["FIN", "BAJA"])
                wrong_fin = np.in1d(aux_df["Movimiento"].values, ["FIN", "BAJA"])
                sp = np.delete(sp, wrong_fin, axis=0)
                if not sp.size:
                    continue
            filt_split.append(sp)
        filt_split = sorted(filt_split, key=lambda x: x[-1][col_fecha])

        return filt_split, t_cols

In [ ]:
df["mov_ord"] = df["Movimiento"].apply(mov_sorter.get)
df_logs = df.sort_values(by=["NTécnico", "Fecha", "mov_ord"])

In [ ]:

df_split, t_cols = splitTrainsByDate(
            df=df_logs,
            # platform=via,
            hour_diff=12,
            filter_mov=False,
        )

In [ ]:
producto = xsiv[["NTécnico",'CategoríaCirculación', 'Producto', 'Empresa']]

In [ ]:
producto

In [ ]:
df_ts = [] 
if not df_split:
    display("no existe trenes")
for df_t in df_split:
    if df_t.empty:
        display("esta vacio")
    df_t = pd.merge(
        producto,
        df_t,
        on=["NTécnico"],
        how = "right"
    )
    df_t.drop_duplicates(subset=["NTécnico", "Fecha", "Elemento"],inplace=True)
    cols = list(df_t.columns)
    if "Fecha" in cols:
        cols.insert(0, cols.pop(cols.index("Fecha")))
        df_t = df_t[cols]
    df_t["tdiff"] = df_t["Fecha"].diff().dt.total_seconds()
    df_t["mdiff"] = df_t["mov_ord"].diff()
    df_t = df_t.reset_index(drop=True).reset_index()
    df_t["prev_index"] = df_t["index"].shift(fill_value=-1)
    for i, row in df_t[1:].iterrows():
        if (row["tdiff"] < 5) & (row["mdiff"] < 0):
            row = row.copy()
            aux_row = df_t.loc[row["prev_index"]].copy()
            mdate = [row["Fecha"], aux_row["Fecha"]]
            aux_row["Fecha"] = max(mdate)
            row["Fecha"] = min(mdate)
            df_t.loc[row["prev_index"]] = row
            df_t.loc[i] = aux_row
            
    df_ts.append(df_t)

In [ ]:
# datagrama = next(
#     (d for d in df_ts if "NTécnico" in d.columns and (d["NTécnico"] == "27093").any()),
#     None
# )
# datagrama

In [ ]:

# for idx, df_t in enumerate(df_ts):
#     df_t = df_t.copy()
#     nuevas_filas = []
#     for i in range(1, len(df_t)):
#         if df_t.loc[i, "Elemento"] != df_t.loc[i-1, "Elemento"]:
#             fila_baja = df_t.loc[i-1].copy()
#             fila_baja["Movimiento"] = "SALIDA"
#             fila_baja["Elemento"] = df_t.loc[i-1, "Elemento"]
#             fila_baja["Fecha"] = df_t.loc[i, "Fecha"]  
#             fila_baja.name = df_t.index.max() + len(nuevas_filas) + 1
#             nuevas_filas.append(fila_baja)
#     if nuevas_filas:
#         df_t = pd.concat([df_t, pd.DataFrame(nuevas_filas)])
#         df_t = df_t.sort_values("index")
#         df_t = df_t.drop(columns = ["index"])
#         df_t.reset_index(drop=True, inplace= True)
#         df_t.sort_values(by=["Fecha"], inplace=True)
#         df_t.reset_index(drop=True, inplace= True)
#     df_ts[idx] = df_t

In [ ]:
topo = getEstacionamientos(["18000"])

In [ ]:
map_estacion_elemento = (
        topo[["Código", "Vía"]]
        .dropna()
        .groupby("Código")
        .agg(lambda x: {el: [] for el in set(x)})
        .to_dict()["Vía"]
    )

In [ ]:
map_estacion_elemento

In [ ]:
# def rellanarSalida(tren:str, codigo:str):
#     s= xsiv[xsiv["NTécnico"] == tren].copy()
#     s.sort_values(by=["Secuencia","Fecha"],inplace=True)
#     s.reset_index(drop=True,inplace = True)
#     idx = s.index[s["Código"] == codigo]
#     if idx.empty:
#         return 
#     idx_start = idx[0]
#     siguiente = s.loc[idx_start+1:]
#     fila = siguiente[siguiente["Código"] != "18000"].head(1)
#     display("fila", fila)
#     if fila.empty:
#         return 
#     # fila = fila.iloc[0]
#     fecha = fila["Fecha"].iloc[0]
#     fecha = fecha.strftime("%Y-%m-%d %H:%M:%S")
#     return fecha


In [ ]:
# datagrama = next(
#     (d for d in df_ts if "NTécnico" in d.columns and (d["NTécnico"] == "27093").any()),
#     None
# )

In [ ]:
# df_ts[0]

In [ ]:
# idx_salida = datagrama[datagrama["Movimiento"] == "SALIDA"].index
# idx_llegada = datagrama[datagrama["Movimiento"] == "LLEGADA"].index
# if not idx_salida.empty and not idx_llegada.empty:
#     ultima_salida = datagrama.loc[idx_salida[-1], "Elemento"]
#     ultima_llegada = datagrama.loc[idx_llegada[-1], "Elemento"]
#     if str(ultima_llegada) != str(ultima_salida):
#         tren = datagrama.loc[idx_llegada[-1], "NTécnico"]
#         codigo = datagrama.loc[idx_llegada[-1], "Código"]
#         fecha = rellanarSalida(tren, codigo)
#         fecha = pd.to_datetime(fecha)
#         display(fecha)
#         # idx_ult_llegada = idx_llegada[-1]
#         # new_row = datagrama.loc[idx_ult_llegada].copy()
#         # new_row["Fecha"] = fecha
#         # new_row["Movimiento"]="SALIDA"
#         # new_row_df = new_row.to_frame().T
#         # train_day= pd.concat([datagrama,new_row_df])
#         # df_ts[idx] = train_day
            


In [ ]:
# for idx, train_day in enumerate(df_ts):
    # idx_salida = train_day[train_day["Movimiento"] == "SALIDA"].index
    # idx_llegada = train_day[train_day["Movimiento"] == "LLEGADA"].index
    # if not idx_salida.empty and not idx_llegada.empty:
    #     ultima_salida = train_day.loc[idx_salida[-1], "Elemento"]
    #     ultima_llegada = train_day.loc[idx_llegada[-1], "Elemento"]
    #     if str(ultima_llegada) != str(ultima_salida):
    #         tren = train_day.loc[idx_llegada[-1], "NTécnico"]
    #         codigo = train_day.loc[idx_llegada[-1], "Código"]
    #         fecha = rellanarSalida(tren, codigo)
    #         fecha = pd.to_datetime(fecha)
    #         idx_ult_llegada = idx_llegada[-1]
    #         new_row = train_day.loc[idx_ult_llegada].copy()
    #         new_row["Fecha"] = fecha
    #         new_row["Movimiento"]="SALIDA"
    #         new_row_df = new_row.to_frame().T
    #         train_day= pd.concat([train_day,new_row_df])
    #         df_ts[idx] = train_day
            


In [ ]:
topo.rename(columns={"VíaTécnica":"Elemento"}, inplace=True)

In [ ]:
# datagrama_20332 = next(
#     (d for d in df_ts if "NTécnico" in d.columns and (d["NTécnico"] == "20332").any()),
#     None
# )

In [ ]:
for idx,df in enumerate(df_ts):
    df = pd.merge(
    topo,
    df,
    on=["Elemento"],
    how="right"
)
    df = df.dropna(subset=["Vía"]).reset_index(drop=True)
    df.drop(columns="Elemento",inplace=True)
    df.rename(columns={"Vía":"Elemento"},inplace=True)
    df_ts[idx] = df


In [ ]:
apariciones = len(df_ts)
station = "18000"
train = []
for n, train_day in enumerate(df_ts, 1):
  for _, row in train_day.iterrows():
    elemento = row["Elemento"]
    if pd.notna(elemento):
      if elemento not in map_estacion_elemento[station]:
        map_estacion_elemento[station][elemento] = []
      map_estacion_elemento[station][elemento].append(row)


In [ ]:

datos = []
cols = (
    # Trenes
    ["T1", "T2", "T_seq", "ProductoT1", "ProductoT2", "EF", "Elemento"]
    # Planificación
    + ["LlegadaPlanificada", "SalidaPlanificada", "OcupaciónPlanificada"]
    # Secuencias de movimientos
    + ["Movimiento", "Mov_seq", "full_seq", "cambio_vía"]
    # Anticipación
    + [
        "Anticipación",
        "AnticipaciónPlataforma",
        "AnticipaciónSalida",
        "AnticipaciónAproximación",
        "AnticipaciónLlegada",
    ]
    # Ocupación
    + ["InicioOcupación", "FinOcupación", "Ocupación"]
    # # Tiempos totales
    # + ["Inicio", "Fin", "TiempoTotal"]
    # Todos los movimientos posibles
    + list(mov_sorter.keys())
        )

In [ ]:
def getMovementType(t1: str, t2: str, mov_ts: str):
        llegada = r"(MANIOBRA→|PREVISIÓN→|APROXIMACIÓN→)*(MANIOBRA→|LLEGADA→)"
        fin = r"(FIN(→)?|ELIMINACIÓN(→)?|SUPRESIÓN(→)?|BAJA(→)?)(MANIOBRA(→)?|CAMBIO_VÍA)*"
        alta = r"(ALTA→)"
        salida = r"(MANIOBRA(→)?|SALIDA(→)?|CAMBIO_VÍA(→)?)"
        # opt_maniobra = r"(MANIOBRA.*?(→)?)"
        if t1 == t2:
            if regex.search(
                rf"^{llegada}+{salida}+$",
                mov_ts,
            ):
                return "PASO"
            elif regex.search(
                rf"^{alta}+{salida}+$",
                mov_ts,
            ):
                return "ORIGEN"
            elif regex.search(
                rf"^{llegada}+{fin}+$",
                mov_ts,
            ):
                return "FIN"
            else:
                return "INCOMPLETO"
        else:
            if regex.search(
                rf"^{llegada}+({fin}+{alta}+)+{salida}+$",
                mov_ts,
            ):
                return "ROTACIÓN"
            elif regex.search(
                rf"^{llegada}+{fin}+{salida}+$",
                mov_ts,
            ):
                return "FIN"
            elif regex.search(
                rf"^({alta}{fin})*{alta}+{salida}+$",
                mov_ts,
            ):
                return "ORIGEN"
            elif regex.search(
                rf"^{llegada}+{alta}+{salida}+$",
                mov_ts,
            ):
                return "RENOMBRADO"
            else:
                return "ROTACIÓN_INCORRECTA"

In [ ]:
map_estacion = map_estacion_elemento["18000"]

In [ ]:
# contador = 0
for Elemento, trains in map_estacion.items():
    if not trains:
        continue
    # *TODO*: Procesar primero cada tren por separado y luego unirlos
    # De esta manera evitamos que haya, por ejemplo, aproximaciones entre la llegada y salida de uno anterior:
    # [llegada tren 1 -> aproximación tren 2 -> salida tren 1] -> [llegada tren 2 -> salida tren 2]
    # pasaría a ser: [llegada tren 1 -> salida tren 1] -> [aproximación tren 2 -> llegada tren 2 -> salida tren 2]

    # Componemos los trenes que han pasado por la vía
    # if contador == 1:
    #     break
    trains_df = [t.to_frame().T if isinstance(t, pd.Series) else t for t in trains]
    df_via = pd.concat(trains_df).sort_values(by=["Fecha", "mov_ord"])
    # via_split = np.split(
    #             df_via,
    #             np.where(
    #                 (df_via["Movimiento"].shift().isin(["SALIDA"]))
    #                 | (
    #                     np.invert(df_via["NTécnico"].shift().eq(df_via["NTécnico"]))
                    
    #                 )
    #             )[0],
    #         )
    via_split = [group for _, group in df_via.groupby("NTécnico")]

    for train_day in via_split:
                if train_day.empty:
                    continue
                info = {c: pd.NA for c in cols}
                info["T1"], info["ProductoT1"] = train_day.iloc[0][
                    ["NTécnico", "Producto"]
                ].values
                info["T2"], info["ProductoT2"] = train_day.iloc[-1][
                    ["NTécnico", "Producto"]
                ].values
                vals = train_day["NTécnico"].values
                seq = [vals[0]]
                for v in vals[1:]:
                    if not v == seq[-1]:
                        seq.append(v)
                info["T_seq"] = "→".join(seq)
                if "Empresa" not in train_day.columns:
                    info["EF"] = setEF(
                        "".join(
                            [
                                el if el and pd.notna(el) else ""
                                for el in [info["ProductoT1"], info["ProductoT2"]]
                            ]
                        )
                    )
                else:
                    empresa = train_day["Empresa"].dropna()
                    if not empresa.empty:
                        info["EF"] = empresa.iloc[0]
                    else:
                        info["EF"] = pd.NA
                    # info["EF"] = train_day["Empresa"].dropna().iloc[0]
                    info["Elemento"] = Elemento
                for mtype, mdate in (
                    train_day[["Movimiento", "Fecha"]]
                    .drop_duplicates(subset=["Movimiento"])
                    .values
                ):
                    info[mtype] = mdate
                    # Secuencia de movimientos registrados
                vals = train_day[["NTécnico", "Movimiento"]].values
                full_seq = [vals[0].tolist()]
                mov_seq = [vals[0][1]]
                for n, v in vals[1:]:
                    # Si el movimiento del tren es igual que el anterior, lo ignoro
                    if n == full_seq[-1][0] and v == full_seq[-1][1]:
                        continue
                    full_seq.append([n, v])
                    mov_seq.append(v)
                info["full_seq"] = full_seq
                info["Mov_seq"] = "→".join(mov_seq)
                if (
                    train_day["Producto"][
                        np.invert(
                            (train_day["Producto"].eq("Material Vacio"))
                            | (train_day["Producto"].apply(isEmpty))
                        )
                    ]
                    .unique()
                    .shape[0]
                    > 1
                ):
                    info["Movimiento"] = "INCORRECTO"
                    info["EF"] = "INCORRECTO"
                else:
                    info["Movimiento"] = getMovementType(
                        info["T1"], info["T2"], info["Mov_seq"]
                    )
                # Planificación
                if "VíaPlanificada" in train_day.columns:
                    plan_arr = train_day["LlegadaPlanificada"].dropna()
                    info["LlegadaPlanificada"] = (
                        plan_arr.iloc[0] if not plan_arr.empty else pd.NaT
                    )
                    plan_dep = train_day["SalidaPlanificada"].dropna()
                    info["SalidaPlanificada"] = (
                        plan_dep.iloc[-1] if not plan_dep.empty else pd.NaT
                    )
                    info["OcupaciónPlanificada"] = (
                        info["SalidaPlanificada"] - info["LlegadaPlanificada"]
                    )

                # Ocupación
                ini_occ = train_day.loc[
                    train_day["Movimiento"].isin(["LLEGADA", "MANIOBRA", "ALTA"]),
                    "Fecha",
                ]
                if not ini_occ.empty:
                    info["InicioOcupación"] = ini_occ.iloc[0]
                end_occ = train_day.loc[
                    train_day["Movimiento"].isin(["SALIDA", "CAMBIO_VÍA", "MANIOBRA"]),
                    "Fecha",
                ]
                if not end_occ.empty:
                    info["FinOcupación"] = end_occ.iloc[-1]
                # print(info)
                info["Ocupación"] = info["FinOcupación"] - info["InicioOcupación"]
                # print(contador)
                # contador = contador + 1


                # Anticipación
                # Si es origen tendrá alta y salida
                m0 = pd.NaT
                m1 = pd.NaT
                # TODO: Cómo se tienen en cuenta las rotaciones?
                if info["Movimiento"] == "ORIGEN":
                    m0 = train_day.loc[train_day["Movimiento"] == "ALTA", "Fecha"].iloc[
                        0
                    ]
                    m1 = train_day.loc[
                        train_day["Movimiento"].isin(
                            ["SALIDA", "CAMBIO_VÍA", "MANIOBRA"]
                        ),
                        "Fecha",
                    ].iloc[0]
                    info["AnticipaciónPlataforma"] = m0
                    info["AnticipaciónSalida"] = m1

                else:
                    # Si no, buscamos aproximación y llegada
                    appr = train_day.loc[
                        train_day["Movimiento"].isin(["APROXIMACIÓN", "PREVISIÓN"]),
                        "Fecha",
                    ]
                    arr = train_day.loc[
                        train_day["Movimiento"].isin(["LLEGADA", "MANIOBRA"]),
                        "Fecha",
                    ]
                    if not appr.empty:
                        m0 = appr.iloc[0]
                    if not arr.empty:
                        m1 = arr.iloc[0]
                    info["AnticipaciónAproximación"] = m0
                    info["AnticipaciónLlegada"] = m1
                info["Anticipación"] = m1 - m0

                # # Tiempos totales
                # info["Inicio"] = train_day["Fecha"].iloc[0]
                # info["Fin"] = train_day["Fecha"].iloc[-1]
                # info["TiempoTotal"] = info["Fin"] - info["Inicio"]

                datos.append(info)
    info_estacion = pd.DataFrame(datos)
    # contador += 1
    


In [ ]:
PRODUCTO_VACIO = [
    "material vacío",
    "material vacio",
    "material vacío ram",
    "material vacio ram",
    "servicio interno",
]
def getProducto(p1: str, p2: str):
    if p1 == p2:
        return p1
    elif isEmpty(p1) or p1.lower().strip() in PRODUCTO_VACIO:
        return p2
    elif isEmpty(p2) or p2.lower().strip() in PRODUCTO_VACIO:
        return p1
    return "?"

In [ ]:
station_historic = info_estacion.reset_index(drop=True)
station_historic["Código"] = station
station_historic["Estación"] = map_cod2name[station]
station_historic["Producto"] = station_historic[
    ["ProductoT1", "ProductoT2"]
].apply(lambda x: getProducto(x["ProductoT1"], x["ProductoT2"]), axis=1)

# Formateamos fechas/horarios
cols_tiempos = ["Anticipación", "OcupaciónPlanificada", "Ocupación"]
cols_tiempos = [c for c in cols_tiempos if c in station_historic.columns]
station_historic[[f"{c} (segundos)" for c in cols_tiempos]] = station_historic[
    cols_tiempos
].map(lambda x: x.total_seconds() if pd.notna(x) else 0)
station_historic[cols_tiempos] = station_historic[cols_tiempos].map(
    lambda x: formatTimedelta(x.total_seconds()) if pd.notna(x) else x
)


In [ ]:

# Buscamos rotaciones que nos han dado
station_historic["RotaciónValidada"] = None
# Hacemos la representación para cada elemento
use_df = station_historic.copy()
# Buscamos fallos para excluirlos y los marcamos en el historico
use_df["Fallo"] = False
fallos = []
for v in use_df["Elemento"].unique():
    ex = use_df[use_df["Elemento"] == v].sort_values(by="InicioOcupación")
    fail = np.where(
        # fin occ. después que inicio occ. siguiente
        (ex["FinOcupación"] > ex["InicioOcupación"].shift(-1))
        # inicio occ. antes que fin occ. anterior
        | (ex["InicioOcupación"] < ex["FinOcupación"].shift(1))
        # fin occ. antes que inicio occ.
        | (ex["InicioOcupación"] > ex["FinOcupación"])
    )[0]
    fallos.extend(ex.iloc[fail].index.tolist())
use_df.loc[fallos, "Fallo"] = True

sname = map_cod2name[station]
fname = f"{station} {map_name2use_name[sname]}"
# fname = f'{station} {sname.lower().replace(" ", "_").replace("-", "_")}'


In [ ]:
station_historic.columns

In [ ]:
def getTramosLibres( df: pd.DataFrame, margen: int = 10, t_min: int = 5):
    """
    Genera un dataframe de tramos libres a partir de un dataframe de ocupación.
    Parámetros:
    -----------
    df: pd.DataFrame
        Tabla de ocupaciones con, al menos:
        - "InicioOcupación"
        - "FinOcupación"
        - "Código"
        - "Vía"
        - "TipoVía"

    margen: int
        Tiempo de seguridad mínimo (en minutos) entre ocupaciones.
    t_min: int
        Duración mínima de ocupación (en minutos).
    """
    df_aux = df.copy()
    # Rellenamos valores vacíos
    df_aux.loc[df_aux["InicioOcupación"].isna(), "InicioOcupación"] = df_aux.loc[
        df_aux["InicioOcupación"].isna(), "FinOcupación"
    ]
    df_aux.loc[df_aux["FinOcupación"].isna(), "FinOcupación"] = df_aux.loc[
        df_aux["FinOcupación"].isna(), "InicioOcupación"
    ]

    # Se incluyen las fechas de las que se dispone en el dataframe
    if df_aux[["InicioOcupación", "FinOcupación"]].dropna().empty:
        return pd.DataFrame(
            columns=[
                "Código",
                "Elemento",
                # "TipoVía",
                "InicioLibre",
                "FinLibre",
                "Libre (segundos)",
                "Libre",
            ]
        )
    min_date = df_aux[["InicioOcupación", "FinOcupación"]].dropna().values.min()
    max_date = df_aux[["InicioOcupación", "FinOcupación"]].dropna().values.max()
    data_free_time = []
    for cod in df_aux["Código"].unique():
        for via in df_aux.loc[df_aux["Código"] == cod, "Elemento"].unique():

            df_via = (
                df_aux[(df_aux["Código"] == cod) & (df_aux["Elemento"] == via)]
                .sort_values("InicioOcupación")
                .reset_index(drop=True)
            )

            # --- Intervalo libre ANTES de la primera ocupación ---
            ini_libre = min_date
            fin_libre = df_via.loc[0, "InicioOcupación"] - timedelta(minutes=margen)

            if fin_libre > ini_libre:
                data_free_time.append([
                    cod, via, ini_libre, fin_libre,
                    (fin_libre - ini_libre).total_seconds()
                ])

            # --- Intervalos libres ENTRE ocupaciones ---
            for i in range(len(df_via) - 1):
                ini_libre = df_via.loc[i, "FinOcupación"] + timedelta(minutes=margen)
                fin_libre = df_via.loc[i + 1, "InicioOcupación"] - timedelta(minutes=margen)

                if fin_libre > ini_libre:
                    data_free_time.append([
                        cod, via, ini_libre, fin_libre,
                        (fin_libre - ini_libre).total_seconds()
                    ])

            # --- Intervalo libre DESPUÉS de la última ocupación ---
            ini_libre = df_via.loc[len(df_via) - 1, "FinOcupación"] + timedelta(minutes=margen)
            fin_libre = max_date

            if fin_libre > ini_libre:
                data_free_time.append([
                    cod, via, ini_libre, fin_libre,
                    (fin_libre - ini_libre).total_seconds()
                ])


    map_tipo_via_tiempo = {"AV": 45, "RC": 25}
    df_free = pd.DataFrame(
        data_free_time,
        columns=[
            "Código",
            "Elemento",
            "InicioLibre",
            "FinLibre",
            "Libre (segundos)",
        ],
    ).dropna()

    df_free["Libre"] = df_free["Libre (segundos)"].apply(formatTimedelta)
    # df_free = df_free[
    #     df_free[["Libre (segundos)", "InicioLibre", "FinLibre"]].apply(
    #         lambda x: (
    #             x["Libre (segundos)"]
    #             >= map_tipo_via_tiempo.get(x["TipoVía"], 0) * 60
    #         )
    #         # & (x["TipoVía"] in map_tipo_via_tiempo.keys())
    #         & (x["FinLibre"] > x["InicioLibre"]),
    #         axis=1,
    #     )
    # ]
    # df_free = df_free[
    #     (df_free["Libre (segundos)"] >= t_min * 60)
    #     & (df_free["FinLibre"] > df_free["InicioLibre"])
    # ]
    df_free[["HoraInicioLibre", "HoraFinLibre"]] = df_free[
        ["InicioLibre", "FinLibre"]
    ].map(lambda x: x.strftime("%H:%M:%S") if x and pd.notna(x) else "")
    return df_free

In [ ]:
from src.visualizacion.ocupacion import visualizacionOcupacionVia
from src.visualizacion.color_maps import (
    color_sorter,
    map_color_anticipacion,
    map_color_EF,
    map_color_ocupacion,
    map_color_saturacion,
    map_shape,
    set_color_anticipacion,
    set_color_ocupacion,
    set_color_saturacion,
    set_name,
    set_shape,
    shape_sorter,
)
from src.visualizacion.utils import BGCOLOR, getSortedPlatforms, setHoverInfo, setLayout

In [ ]:
use_df.columns

In [ ]:
margen = 10
t_min = 5
min_date = pd.to_datetime(
        (pd.to_datetime(use_df["FinOcupación"]).min() - timedelta(hours=0.5)).strftime(
            "%Y-%m-%d %H"
        )
    ) - timedelta(hours=1)
max_date = pd.to_datetime(
        (pd.to_datetime(use_df["FinOcupación"]).max() + timedelta(hours=0.5)).strftime(
            "%Y-%m-%d %H"
        )) + timedelta(hours=1)
df_free = (
    getTramosLibres(use_df, margen=margen, t_min=t_min)
    .sort_values(by=["InicioLibre"])
    .reset_index(drop=True)
)


In [ ]:
use_df[use_df["T_seq"] == "37078"]

In [ ]:
def sortStrNumbers(str_list: list[str]):
    """
    Ordena una lista de strings que contiene números por número.
    """
    str_list = [f"{el}" for el in str_list if regex.search("\d+", f"{el}")]
    return sorted(str_list, key=lambda x: int(regex.search("\d+", x).group()))

In [ ]:
def getSortedPlatforms(vias: list[str], cod: str):
    # Separamos AM de convencional
    am = sortStrNumbers([v for v in vias if "AM" in v])
    rc = sortStrNumbers([v for v in vias if "AM" not in v])
    sorted_platforms = rc + am
    if cod == "17000":
        sorted_platforms = [
            v
            for v in (
                ["1", "2", "3", "4", "5", "6", "7"]
                + ["8", "9B", "9", "10", "10B", "11", "12", "13"]
                + ["14", "15", "16", "17A", "17B", "18A", "18B"]
                + ["19A", "19B", "20", "21", "22A", "22B"]
                + ["23A", "23B", "24A", "24B", "25A", "25B"]
            )
            if v in sorted_platforms
            if v
        ]
    platform_map = {v: i for i, v in enumerate(sorted_platforms)}
    return platform_map



In [ ]:
def addRectTrace(
    inicio,
    fin,
    y_inicio,
    y_fin,
    dy,
    mode,
    color,
    name,
    hover_info,
    showlegend: bool = True,
    opacity: float = 1,
    width: float = 0,
    dash=None,
):
    if mode == "markers":
        x = (inicio or fin,)
        y = (y_inicio or y_fin,)
        fill = None
    else:
        x = (inicio, inicio, fin, fin, inicio)
        y = (y_inicio + dy, y_inicio - dy, y_fin - dy, y_fin + dy, y_inicio + dy)
        fill = "toself"

    trace = go.Scatter(
        x=x,
        y=y,
        mode=mode,
        line=dict(color=color, width=width, dash=dash),
        visible=True,
        fill=fill,
        fillcolor=color,
        hoverinfo="text",
        hovertext=hover_info,
        # hoveron="points+fills",
        hoveron="points",
        textfont=dict(family="calibri", size=18),
        name=name,
        showlegend=showlegend,
        legendgroup=name,
        opacity=opacity,
    )
    return trace

In [ ]:
def incluirLibre(df: pd.DataFrame, platform_map: dict, margen=10, t_min=10):
    """
    margen: int
        Tiempo de seguridad mínimo (en minutos) entre ocupaciones.
    t_min: int
        Duración mínima de ocupación (en minutos).
    """
    if df.empty:
        return []

    df_rep = df.copy()

    # Info para mostrar
    hover_cols_free = [
        "Vía",
        "TipoVía",
        "HoraInicioLibre",
        "HoraFinLibre",
    ]
    df_rep["hover_info"] = setHoverInfo(df_rep, hover_cols_free)

    # Ordenamos las vías
    df_rep.rename(columns={"Elemento":"Vía"}, inplace=True)
    df_rep["Vía_order"] = df_rep["Vía"].apply(platform_map.get)

    opacity = 0.66
    width = 1
    mode = "lines"
    name = f"Libre (>{t_min} min)"

    traces = []
    for i, row in df_rep.iterrows():
        if row["Vía"] not in platform_map:
            continue
        trace = addRectTrace(
            inicio=row["InicioLibre"],
            fin=row["FinLibre"],
            y_inicio=row["Vía_order"],
            y_fin=row["Vía_order"],
            dy=0.35,
            mode=mode,
            color="silver",
            name=name,
            hover_info=row["hover_info"],
            showlegend=True if not i else False,
            opacity=opacity,
            width=width,
        )
        traces.append(trace)
    return traces

In [ ]:
from src.visualizacion.color_maps import (
    color_sorter,
    map_color_anticipacion,
    map_color_EF,
    map_color_ocupacion,
    map_color_saturacion,
    map_shape,
    set_color_anticipacion,
    set_color_ocupacion,
    set_color_saturacion,
    set_name,
    set_shape,
    shape_sorter,
)
from src.visualizacion.utils import BGCOLOR, getSortedPlatforms, setHoverInfo, setLayout

In [ ]:
def addRectTrace(
    inicio,
    fin,
    y_inicio,
    y_fin,
    dy,
    mode,
    color,
    name,
    hover_info,
    showlegend: bool = True,
    opacity: float = 1,
    width: float = 0,
    dash=None,
):
    if mode == "markers":
        x = (inicio or fin,)
        y = (y_inicio or y_fin,)
        fill = None
    else:
        x = (inicio, inicio, fin, fin, inicio)
        y = (y_inicio + dy, y_inicio - dy, y_fin - dy, y_fin + dy, y_inicio + dy)
        fill = "toself"

    trace = go.Scatter(
        x=x,
        y=y,
        mode=mode,
        line=dict(color=color, width=width, dash=dash),
        visible=True,
        fill=fill,
        fillcolor=color,
        hoverinfo="text",
        hovertext=hover_info,
        # hoveron="points+fills",
        hoveron="points",
        textfont=dict(family="calibri", size=18),
        name=name,
        showlegend=showlegend,
        legendgroup=name,
        opacity=opacity,
    )
    return trace


In [ ]:
map_EF_color = {
    "RENFE": "#830065",
    "IRYO": "#DA291C",
    "OUIGO": "#0096CA",
    "INCORRECTO": "khaki",
    "Otro": "mediumblue",
}
color_sorter = {
    c: i
    for i, c in enumerate(
        [
            "#830065",  # pantone 2425c renfe
            "#DA291C",  # pantone 485c iryo
            "#0096CA",  # azul ouigo
            "silver",
            "mediumblue",
            "green",
            "yellow",
            "khaki",
            "orange",
            "red",
            "darkred",
            "black",
        ]
    )
}

map_color_EF = {c: ef for ef, c in map_EF_color.items()}
def set_color_ocupacion(s, criterio="EF"):
    """
    Establece el color de la ocupación en función de un criterio.
    criterio: {"EF", "tiempo"}
    """
    # if criterio == "EF":
    #     if regex.search(
    #         r"(ALVIA|AVANT|AVE|CERCANIAS|INTERCITY|MD|REGIONAL EXPRES|TALGO|AVLO)", s
    #     ):
    #         return "#830065"
    #     if regex.search(r"(IRYO)", s):
    #         return "#DA291C"
    #     elif regex.search(r"(OUIGO)", s):
    #         return "#0096CA"
    #     else:
    #         return "mediumblue"
    if criterio == "EF":
        return map_EF_color.get(s, "mediumblue")

    elif criterio == "tiempo":
        if s < 5:
            return "mediumblue"
        elif s <= 120:
            return "green"
        elif s < 180:
            return "orange"
        elif s <= 300:
            return "red"
        elif s > 300:
            return "darkred"
        else:
            return "black"
map_color_ocupacion = {
    "mediumblue": "<5 segundos",
    "green": "<120 segundos",
    # "yellow": "<180 segundos",
    "orange": "<180 segundos",
    "red": "<300 segundos",
    "darkred": ">300 segundos",
    "black": "Error",
}
def setHoverInfo(df: pd.DataFrame, hover_cols: list[str]):
    hover_cols = [c for c in hover_cols if c in df.columns]
    # Máxima longitud de nombre de columnas
    c_len = max([len(c) for c in hover_cols]) + 2
    # Fijamos la longitud máxima del valor, si se supera, hay un salto de línea
    fd_len = min(
        df[hover_cols]
        .fillna("")
        .map(lambda x: len(f"{x}"), na_action="ignore")
        .max()
        .max(),
        25,
    )

    def fitLen(value, sep):
        """
        Ajusta el tamaño del recuadro que muestra la info
        """
        if isinstance(value, list):
            row_sep = []  # lista de filas
            split = []  # elementos de cada fila
            for el in value:
                # Si la concatenación es menor que fd_len lo unimos
                if len(regex.sub("<.+?>", "", sep.join(split + [el]))) < fd_len:
                    split.append(el)
                else:
                    # sino, añadimos la fila y creamos una nueva
                    if split:
                        row_sep.append(sep.join(split))
                    split = [el]
            row_sep.append(sep.join([f"{el.strip()}" for el in split]))
            fit = []
            # Para cada fila, rellenamos el "vacío" a su izquierda para que quede justificado a la derecha
            for i, el in enumerate(row_sep):
                # El título de la columna va en la primera fila
                if i:
                    fit.append(
                        f"<b>{'':<{c_len}}</b>{sep+el:>{fd_len+len(''.join(regex.findall('<.+?>', el)))}}"
                    )
                else:
                    fit.append(
                        f"{el:>{fd_len+len(''.join(regex.findall('<.+?>', el)))}}"
                    )
            return "<br>".join(fit)
        elif isinstance(value, str):
            # Separamos por palabras
            if "→" in value:
                fit = fitLen(regex.split(r"→+", value), sep="→")
            else:
                fit = fitLen(regex.split(r"\s+", value), sep=" ")
            # fit = regex.sub(r"(?<=<br>.+)\s(?=\w)", "→", fit.replace(",", "→"))
            return fit
        return (
            f"{str(value):>{fd_len+len(''.join(regex.findall('<.+?>', str(value))))}}"
        )

    hov_info = df.fillna("").apply(
        lambda row: "<br>".join(
            [f"<b>{c:<{c_len}}</b>{fitLen(row[c], sep=' ')}" for c in hover_cols]
        ),
        axis=1,
    )
    if hov_info.empty:
        return None
    return hov_info


def incluirOcupaciones(df: pd.DataFrame, platform_map: dict, criterio: str = "EF"):
    if df.empty:
        return []

    df_rep = (
        df.copy()
        # .sort_values(by=["LlegadaPlanificada"])
        .reset_index(drop=True)
    )

    # Definimos el color
    if criterio == "EF":
        map_color = map_color_EF
        if "EF" not in df_rep.columns:
            df_rep["EF"] = df_rep["Producto"].apply(setEF)
        df_rep["color"] = df_rep["EF"].apply(
            lambda x: set_color_ocupacion(x, criterio=criterio)
        )
        # Si el movimiento es incorrecto, independientemente de EF, lo marcamos como tal
        df_rep.loc[df_rep["Movimiento"] == "INCORRECTO", "color"] = set_color_ocupacion(
            "INCORRECTO", criterio=criterio
        )
    elif criterio == "tiempo":
        map_color = map_color_ocupacion
        df_rep["color"] = df_rep["Ocupación (segundos)"].apply(
            lambda x: (
                set_color_ocupacion(x, criterio=criterio)
                if pd.notna(x)
                else set_color_ocupacion(0, criterio=criterio)
            )
        )

    # Info para mostrar
    df_rep[["HoraInicioOcupación", "HoraFinOcupación"]] = df_rep[
        ["InicioOcupación", "FinOcupación"]
    ].map(lambda x: x.strftime("%H:%M:%S") if pd.notna(x) else "")
    hover_cols = [
        "NTécnico",
        "Mov_seq",
        "EF",
        "Producto",
        "Movimiento",
        "HoraInicioOcupación",
        "HoraFinOcupación",
        "Ocupación",
        "Vía",
        # "TipoVía",
    ]
    df_rep["hover_info"] = setHoverInfo(df_rep, hover_cols)

    # Ordenamos las vías
    df_rep["Vía_order"] = df_rep["Vía"].apply(platform_map.get)

    traces = []
    used_names = []
    for c in sorted(df_rep["color"].unique(), key=color_sorter.get):
        df_aux = df_rep[df_rep["color"] == c]
        name = set_name(color=c, map_color=map_color)

        trace = addRectTrace(
            inicio=(None,),
            fin=(None,),
            y_inicio=0,
            y_fin=0,
            dy=0,
            mode="lines",
            color=c,
            name=name,
            hover_info=None,
            showlegend=True,
            opacity=1,
            width=0,
        )
        if name not in used_names:
            used_names.append(name)
        traces.append(trace)

        for _, row in df_aux.sort_values(by=["InicioOcupación"]).iterrows():
            if row["Vía"] not in platform_map:
                continue
            if row["Ocupación (segundos)"] < 5:
                inicio = row["InicioOcupación"] or row["FinOcupación"]
                fin = row["InicioOcupación"] or row["FinOcupación"]
                opacity = 1
                dy = 0
                mode = "markers"
                width = 0
            else:
                inicio = row["InicioOcupación"]
                fin = row["FinOcupación"]
                dy = 0.15
                opacity = 1
                mode = "lines"
                width = 1

            trace = addRectTrace(
                inicio=inicio,
                fin=fin,
                y_inicio=row["Vía_order"],
                y_fin=row["Vía_order"],
                dy=dy,
                mode=mode,
                color=c,
                name=name,
                hover_info=row["hover_info"],
                showlegend=True if name not in used_names else False,
                opacity=opacity,
                width=width,
            )
            traces.append(trace)
            if name not in used_names:
                used_names.append(name)
    return traces


In [ ]:
def visualizacionOcupacionVia(
    df: pd.DataFrame,
    title: str = "",
    df_free: pd.DataFrame = None,
    margenes: pd.DataFrame = None,
    show_plan: bool = True,
    show_fail: bool = True,
):
    # En principio el criterio de colores es la empresa ferroviaria
    criterio = "EF"

    if df is None or df.empty:
        return

    traces = []
    use_df = df.copy().replace([pd.NA], [None])
    # Ordenamos las vías por número
    platform_map = getSortedPlatforms(
        use_df["Vía"].dropna().unique(), use_df["Código"].iloc[0]
    )
    # Pintamos tiempos libres
    if df_free is not None and not df_free.empty:
        traces.extend(incluirLibre(df_free, platform_map, margen=10, t_min=40))
    # # Pintamos los fallos
    # if show_fail:
    #     traces.extend(incluirFallos(use_df[use_df["Fallo"]], platform_map))
    # # Incluir planificación
    # if show_plan and "Ocupación planificada (segundos)" in use_df.columns:
    #     traces.extend(
    #         incluirPlanificacion(use_df, platform_map=platform_map, criterio=criterio)
    #     )
    # # Incluir planificación
    # if margenes is not None and not margenes.empty:
    #     traces.extend(incluirMargenes(margenes, platform_map=platform_map, margen=10))
    # # Pintamos los cambios de vía
    # if criterio == "EF":
    #     traces.extend(
    #         incluirCambioVia(
    #             use_df.dropna(subset="cambio_vía"), platform_map=platform_map
    #         )
    #     )
    # Pintamos las ocupaciones normales
    traces.extend(
        incluirOcupaciones(
            use_df[~use_df["Fallo"]], platform_map=platform_map, criterio=criterio
        )
    )

    # Creamos el layout
    min_date = pd.to_datetime(
        (pd.to_datetime(use_df["FinOcupación"]).min() - timedelta(hours=0.5)).strftime(
            "%Y-%m-%d %H"
        )
    ) - timedelta(hours=1)
    max_date = pd.to_datetime(
        (pd.to_datetime(use_df["FinOcupación"]).max() + timedelta(hours=0.5)).strftime(
            "%Y-%m-%d %H"
        )
    ) + timedelta(hours=1)
    layout = setLayout("togglegroup", title, platform_map, x_range=(min_date, max_date))
    # layout = None

    fig = go.Figure(data=traces, layout=layout)

    return fig

In [ ]:
use_df.rename(columns={"Elemento":"Vía"}, inplace=True)

In [ ]:
use_df.rename(columns={"T_seq":"NTécnico"}, inplace=True)

In [ ]:

fig_occ = visualizacionOcupacionVia(
        use_df,
        title=f"Ocupación de vías en <b>{fname}</b>",
        df_free=df_free,
        show_fail=True,
        show_plan=True,
    )

In [ ]:
use_df


In [ ]:
fig_occ

In [ ]:
output_dir = Path(r"C:\Users\xiangzhou.zhang\Documents\Codigo\LogProcess\output\test")
output_path = f"{output_dir}/ocupacion_vias_Test_jtct_nuevo3.html"
fig_occ.write_html(output_path)

In [ ]:
historico_pro